In [ ]:
OPEN_ROUTER_KEY = "OPEN_ROUTER_KEY"
OPEN_API_KEY = "OPEB_API_KEY"

## installation

In [ ]:
!pip install -U \
  "langchain==0.3.*" \
  "langchain-community==0.3.*" \
  "langchain-openai==0.3.*" \
  "langchain-experimental==0.3.4" \
  "sentence-transformers>=3,<4" \
  "faiss-cpu>=1.8"


In [ ]:
!pip install pythainlp

## import

In [ ]:
import os
import re
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Callable, Union
from dataclasses import dataclass
import json

# LangChain imports
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker as LC_SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.retrievers import BaseRetriever

# Thai NLP
from pythainlp.tokenize import word_tokenize


## elective process

In [ ]:
# --- electives.py ---
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

PROGRAM_RE   = re.compile(r"^\s*หลักสูตร[ \t]+(?P<program>[^\r\n]+?)\s*$", re.MULTILINE)
SEM_TITLE_RE = re.compile(
    r"^\s*ปีที่\s*(?P<year>[0-9๐-๙]+)\s*ภาคการศึกษาที่\s*(?P<sem>[0-9๐-๙]+)\s*$",
    re.MULTILINE
)


@dataclass(frozen=True)
class Key:
    program: str
    year: int
    sem: int

class ElectiveCatalog:
    """
    เก็บวิชาเลือกด้วยคีย์ (program, year, sem) -> List[str]
    """
    def __init__(self, index: Optional[Dict[Key, List[str]]] = None):
        self.index: Dict[Key, List[str]] = index or {}

    @staticmethod
    def _normalize_program(s: str) -> str:
        return re.sub(r"\s+", " ", s.strip())

    @classmethod
    def from_text(cls, text: str) -> "ElectiveCatalog":
        """
        รับ blob วิชาเลือกตาม format ตัวอย่าง แล้วแตกเป็นดัชนี
        รูปแบบ:
            หลักสูตร วิทยาการคอมพิวเตอร์
            ปีที่ 2 ภาคการศึกษาที่ 1
            <รายวิชา...>
            [ว่างบรรทัด/หรือบล็อคใหม่เริ่มด้วย 'หลักสูตร ...']
        """
        blocks = re.split(r"(?=^\s*หลักสูตร\s.*$)", text, flags=re.MULTILINE)
        index: Dict[Key, List[str]] = {}

        for b in blocks:
            if not b.strip():
                continue

            m_prog = PROGRAM_RE.search(b)
            if not m_prog:
                continue
            program = cls._normalize_program(m_prog.group("program"))

            segs = list(SEM_TITLE_RE.finditer(b))
            for i, seg in enumerate(segs):
                y = int(seg.group("year"))
                s = int(seg.group("sem"))
                start = seg.end()
                end = segs[i+1].start() if i+1 < len(segs) else len(b)
                body = b[start:end].strip()

                lines = []
                for line in body.splitlines():
                    line = line.strip()
                    if not line:
                        continue
                    if PROGRAM_RE.match(line) or SEM_TITLE_RE.search(line):
                        continue
                    lines.append(line)

                key = Key(program=program, year=y, sem=s)
                if lines:
                    index.setdefault(key, []).extend(lines)

        return cls(index=index)

    def get(self, program: str, year: int, sem: int) -> List[str]:
        key = Key(program=self._normalize_program(program), year=year, sem=sem)
        return self.index.get(key, [])

def extract_year_sem_from_title(title: Optional[str]) -> Optional[Tuple[int, int]]:
    if not title:
        return None
    m = SEM_TITLE_RE.search(title)
    if not m:
        return None
    return int(m.group("year")), int(m.group("sem"))

def merge_electives_into_chunks(
    chunks: List[Dict[str, str]],
    program_name: str,
    catalog: ElectiveCatalog,
    header_label_for_append: str = "วิชาเลือกเพิ่มเติม"
) -> List[Dict[str, str]]:
    """
    รับ chunks [{title, content}, ...] แล้วถ้า title ใดเป็น 'ปีที่ X ภาคการศึกษาที่ Y'
    จะ concat รายวิชาเลือกจาก catalog เข้าท้าย content ของ chunk นั้น
    """
    merged: List[Dict[str, str]] = []
    for ch in chunks:
        title = ch.get("title")
        content = ch.get("content", "").rstrip()

        ys = extract_year_sem_from_title(title)
        if ys:
            y, s = ys
            electives = catalog.get(program=program_name, year=y, sem=s)
            if electives:
                block = "\n\n" + header_label_for_append + ":\n" + "\n".join(f"- {c}" for c in electives)
                content = content + block
        merged.append({"title": title, "content": content})
    return merged

def infer_program_from_text(text: str, catalog: ElectiveCatalog) -> Optional[str]:
    """
    เดาชื่อหลักสูตรจากเนื้อหา section โดย:
    1) หา 'หลักสูตร <ชื่อ>' ที่ปรากฏใน text
    2) ถ้าเจอ ให้เลือกชื่อที่ 'ใกล้เคียง' กับรายการใน catalog (exact ก่อน, ถ้าไม่เจอลอง contains)
    """
    # ดึง candidate จากข้อความ
    m = re.search(r"หลักสูตร\s*([^\n\r]+)", text)
    if not m:
        return None
    candidate = re.sub(r"\s+", " ", m.group(1).strip())

    # exact match ก่อน
    for p in catalog.programs():
        if p == candidate:
            return p

    # ถ้าไม่ exact ลอง contains/partial แบบง่าย ๆ
    cand_norm = candidate.replace(" ", "")
    best = None
    for p in catalog.programs():
        pn = p.replace(" ", "")
        if pn in cand_norm or cand_norm in pn:
            best = p
            break
    return best

## document process

In [ ]:
@dataclass
class CourseConfig:
    """Configuration for each หลักสูตร"""
    name: str
    file_path: str
    offset: int = 0
    section_titles: List[str] = None
    selected_headers: List[str] = None
    subheader_map: Dict[str, List[str]] = None
    chunking_plan: Dict[str, Dict[str, Any]] = None

    program_name: Optional[str] = None
    electives_catalog: Optional[ElectiveCatalog] = None
    electives_label: str = "วิชาเลือกเพิ่มเติม"

In [ ]:
class DocumentProcessor:
    """Handles document processing for multiple courses"""

    def __init__(self):
        self.embeddings = None
        self.sem_splitter = None
        self.initialize_models()

    def initialize_models(self):
        """Initialize embedding models and semantic splitter"""
        # self.embeddings = HuggingFaceEmbeddings(
        #     model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        #     encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
        # )
        self.embeddings = OpenAIEmbeddings(
          model="text-embedding-3-large",
          openai_api_key=OPEN_API_KEY
      )

        self.sem_splitter = LC_SemanticChunker(
            self.embeddings,
            breakpoint_threshold_type="percentile",
            breakpoint_threshold_amount=80.0,
            min_chunk_size=100,
        )

class BaseSplitter:
    """Base class for document splitters"""
    method_name = "base"

    def split(self, toc_header: str, content: str, **kwargs) -> List[Dict[str, str]]:
        raise NotImplementedError

class SemanticSplitter(BaseSplitter):
    method_name = "semantic"

    def split(self, toc_header: str, content: str, **kwargs) -> List[Dict[str, str]]:
        return [{"title": None, "content": content.strip()}]

class SubTocSplitter(BaseSplitter):
    method_name = "subtoc"

    def split(self, toc_header: str, content: str, **kwargs) -> List[Dict[str, str]]:
        markers: Optional[List[str]] = kwargs.get("markers")
        if not markers:
            return [{"title": None, "content": content.strip()}]

        hits: List[Tuple[int, str]] = []
        for m in markers:
            rgx = self._fuzzy_header_regex(m)
            match = rgx.search(content) or re.search(re.escape(m), content)
            if match:
                hits.append((match.start(), m))

        if not hits:
            return [{"title": None, "content": content.strip()}]

        hits.sort()
        parts = []
        for i, (start, title) in enumerate(hits):
            end = hits[i + 1][0] if i + 1 < len(hits) else len(content)
            parts.append({"title": title, "content": content[start:end].strip()})
        return parts

    def _fuzzy_header_regex(self, s: str) -> re.Pattern:
        words = re.split(r"\s+", s.strip())
        joined = r"\s+".join(map(re.escape, words))
        return re.compile(rf"(?m)^{joined}")

class RegexSplitter(BaseSplitter):
    method_name = "regex"

    def split(self, toc_header: str, content: str, **kwargs) -> List[Dict[str, str]]:
        pattern = kwargs.get("pattern", "")
        remove_patterns = kwargs.get("remove_patterns", []) or []
        remove_between = kwargs.get("remove_between", []) or []
        keep_title = kwargs.get("keep_title", True)
        skip_when_no_match = kwargs.get("skip_when_no_match", True)
        only_first = kwargs.get("only_first", False)

        if not pattern:
            return [{"title": None, "content": self._clean_text(content.strip(), remove_patterns, remove_between)}]

        try:
            rgx = re.compile(pattern, flags=re.MULTILINE | re.DOTALL)
            hits = [(m.start(), m.group(0)) for m in rgx.finditer(content)]

            if not hits:
                return [] if skip_when_no_match else [{"title": None, "content": self._clean_text(content.strip(), remove_patterns, remove_between)}]

            parts = []
            if only_first:
                start, title_text = hits[0]
                end = len(content)
                sec = content[start:end].strip()
                sec = self._clean_text(sec, remove_patterns, remove_between)
                parts.append({"title": title_text.strip() if keep_title else None, "content": sec})
                return parts

            for i, (start, title_text) in enumerate(hits):
                end = hits[i + 1][0] if i + 1 < len(hits) else len(content)
                sec = content[start:end].strip()
                sec = self._clean_text(sec, remove_patterns, remove_between)
                parts.append({"title": title_text.strip() if keep_title else None, "content": sec})

            return parts
        except re.error as e:
            print(f"Error: Invalid regex '{pattern}': {e}")
            return [{"title": None, "content": self._clean_text(content.strip(), remove_patterns, remove_between)}]

    def _clean_text(self, text: str, remove_patterns: List[str], remove_between: List[Dict[str, Any]]) -> str:
        for p in remove_patterns:
            text = re.sub(p, "", text, flags=re.MULTILINE | re.DOTALL)

        for r in remove_between:
            start, end = r.get("start"), r.get("end")
            inclusive = r.get("inclusive", True)
            if not start or not end:
                continue
            rgx = f"{start}.*?{end}" if inclusive else f"(?<={start}).*?(?={end})"
            text = re.sub(rgx, "", text, flags=re.MULTILINE | re.DOTALL)
        return text


## key extraction

In [ ]:
from typing import List, Dict, Tuple, Any
import numpy as np
from sentence_transformers import SentenceTransformer
from pythainlp import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from functools import lru_cache
import asyncio
import aiohttp
import json
from concurrent.futures import ThreadPoolExecutor
import time

class EmbeddingKeywordExtractor:
    def __init__(self, use_local_model: bool = False):
        """
        Initializes the EmbeddingKeywordExtractor.

        Args:
            use_local_model (bool): If True, uses a local SentenceTransformer model
                                   for faster, offline processing. If False (default),
                                   uses the OpenAI API.
        """
        # --- Model Initialization ---
        if not use_local_model:
            # Note: Make sure langchain-openai is installed: pip install langchain-openai
            self.model = OpenAIEmbeddings(
                model="text-embedding-3-small",
                openai_api_key=OPEN_API_KEY
            )
            self._encode_method = self._encode_openai
        else:
            # Use a local multilingual model for Thai
            self.model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
            self._encode_method = self._encode_local

        # --- Caching and Pre-computation ---
        # FIX: Initialize the cache to prevent AttributeError
        self.embedding_cache: Dict[str, np.ndarray] = {}

        headers = self.get_default_headers_text()
        self.headers = headers

        # Precompute normalized embeddings for headers once at initialization
        print("Pre-computing header embeddings...")
        self.H = self._encode_method(headers)
        print("Header embeddings ready.")

        # --- Optional: For potential future parallelization ---
        self.executor = ThreadPoolExecutor(max_workers=4)

    # ---------- Utility ----------
    def _normalize(self, arr: np.ndarray) -> np.ndarray:
        """L2-normalizes a 2D numpy array along its rows."""
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        norms[norms == 0] = 1  # Avoid division by zero
        return arr / norms

    def _encode_openai(self, texts: List[str], normalize: bool = True) -> np.ndarray:
        """
        Encodes texts using OpenAI's API with a caching layer to avoid redundant calls.
        """
        if isinstance(texts, str):
            texts = [texts]

        # 1. Check cache for all texts
        uncached_texts = []
        uncached_indices = []
        results = [None] * len(texts)

        for i, text in enumerate(texts):
            if text in self.embedding_cache:
                results[i] = self.embedding_cache[text]
            else:
                uncached_texts.append(text)
                uncached_indices.append(i)

        # 2. Fetch embeddings only for texts not in cache
        if uncached_texts:
            # Use embed_documents for batch processing
            new_embeddings = np.array(self.model.embed_documents(uncached_texts))
            if normalize:
                new_embeddings = self._normalize(new_embeddings)

            # 3. Update cache and results
            for text, embedding in zip(uncached_texts, new_embeddings):
                self.embedding_cache[text] = embedding

            for idx, embedding in zip(uncached_indices, new_embeddings):
                results[idx] = embedding

        return np.array(results)

    def _encode_local(self, texts: List[str], normalize: bool = True) -> np.ndarray:
        """Encodes texts using a local SentenceTransformer model."""
        if isinstance(texts, str):
            texts = [texts]
        embs = self.model.encode(texts)
        return self._normalize(embs) if normalize else embs

    # A single wrapper to handle encoding regardless of the model
    def _encode(self, texts: List[str], normalize: bool = True) -> np.ndarray:
        return self._encode_method(texts, normalize)

    # ---------- Tokenization ----------
    def tokenize(self, text: str) -> List[str]:
        return word_tokenize(text or "", engine="newmm", keep_whitespace=False)

    def make_ngrams(self, tokens: List[str], n_max: int = 3) -> List[str]:
        """
        Generates n-grams from tokens with optimizations to reduce candidates.
        """
        ngrams = set()
        token_count = len(tokens)

        # Limit the number of tokens to prevent excessive computation
        max_tokens = min(token_count, 100)
        tokens = tokens[:max_tokens]

        # Common Thai stop words to filter out
        stop_words = {"และ", "ของ", "ใน", "ที่", "เป็น", "กับ", "มี", "จะ", "ได้", "ต้อง", "คือ", "นี้", "โดย", "แล้ว", "เพื่อ", "ซึ่ง", "สำหรับ", "อย่าง"}

        for n in range(1, min(n_max + 1, max_tokens) + 1):
            for i in range(max_tokens - n + 1):
                phrase = "".join(tokens[i:i+n])
                # Filter out very short phrases and common stop words
                if len(phrase) > 2 and phrase not in stop_words:
                    ngrams.add(phrase)

        return list(ngrams)

    # ---------- Tag extraction ----------
    def header_tags_for_chunk(self, text: str, top_k: int = 3, tau: float = 0.3) -> Tuple[List[str], Dict[str, float]]:
        if not text:
            return [], {}

        # Get embedding for the text chunk
        c = self._encode([text])[0]

        # Calculate cosine similarity with all pre-computed header vectors
        sims = self.H @ c  # Efficient dot product for normalized vectors

        # Get top_k most similar headers
        order = np.argsort(-sims)
        top_pairs = [(self.headers[i], float(sims[i])) for i in order[:top_k] if sims[i] >= tau]

        return [h for h, _ in top_pairs], dict(top_pairs)

    def filter_keywords_by_tags(self, keywords: List[str], tags: List[str], top_m: int = 12, tau_kw: float = 0.35) -> List[str]:
        if not keywords:
            return []
        if not tags:
            return keywords[:top_m]

        # Create a single vector representing all tags
        tag_vec = self._encode([" ".join(tags)])[0]

        # Get all keyword vectors in one batch
        kw_vecs = self._encode(keywords)

        # Calculate similarity to the tag vector
        sims = kw_vecs @ tag_vec
        order = np.argsort(-sims)

        # Filter and return top keywords
        keep = [keywords[i] for i in order if sims[i] >= tau_kw]
        return (keep or [keywords[i] for i in order])[:top_m]

    # ---------- Keyword extraction ----------
    def extract_keywords(self, text: str, top_k: int = 10) -> List[str]:
        if not text:
            return []

        toks = self.tokenize(text)
        candidates = self.make_ngrams(toks)
        if not candidates:
            return []

        # Get document embedding
        doc_vec = self._encode([text])[0]

        # Get all candidate embeddings in one batch
        cand_vecs = self._encode(candidates)

        # Calculate similarity scores
        sims = cand_vecs @ doc_vec
        order = np.argsort(-sims)

        # Deduplicate and select top keywords
        seen, keywords = set(), []
        for i in order:
            k = candidates[i]
            kl = k.lower()
            if kl not in seen:
                seen.add(kl)
                keywords.append(k)
            if len(keywords) >= top_k:
                break
        return keywords

    def extract_keywords_batch(self, texts: List[str], top_k: int = 10) -> List[List[str]]:
        """
        A batch version of extract_keywords for processing multiple texts at once.
        This is much faster than calling extract_keywords repeatedly.
        """
        if not texts:
            return []

        # 1. Tokenize and generate candidates for all texts
        all_tokens = [self.tokenize(text) for text in texts]
        all_candidates = [self.make_ngrams(toks) for toks in all_tokens]

        # 2. Get document embeddings for all texts in one API call
        doc_vecs = self._encode(texts)

        # 3. Process each text's candidates
        results = []
        for i, (candidates, doc_vec) in enumerate(zip(all_candidates, doc_vecs)):
            if not candidates:
                results.append([])
                continue

            # Get all candidate embeddings for this specific document
            cand_vecs = self._encode(candidates)

            sims = cand_vecs @ doc_vec
            order = np.argsort(-sims)

            seen, keywords = set(), []
            for j in order:
                k = candidates[j]
                kl = k.lower()
                if kl not in seen:
                    seen.add(kl)
                    keywords.append(k)
                if len(keywords) >= top_k:
                    break
            results.append(keywords)

        return results

    def get_default_headers_text(self) -> str:
        """Get default headers text for Thai curriculum documents"""
        return [
    # หมวดที่ 1 ข้อมูลทั่วไป
    "1. รหัสและชื่อหลักสูตร : หลักสูตร ชื่อหลักสูตร ชื่อปริญญา ชื่อเต็ม ชื่อย่อ ไทย อังกฤษ",
    "2. ชื่อปริญญาและสาขาวิชา",
    "4. จำนวนหน่วยกิตที่เรียนตลอดหลักสูตร : หน่วยกิต รวมทั้งหมด",
    "5. รูปแบบของหลักสูตร : หลักสูตร 4 ปี ปริญญาตรี ภาษาที่ใช้ การรับเข้า นักศึกษาไทย ต่างชาติ",

    # หมวดที่ 3 ระบบการจัดการศึกษา การดำเนินการ และโครงสร้างหลักสูตร
    "3.1 ระบบการจัดการศึกษา : ระบบทวิภาค ภาคต้น ภาคปลาย ภาคเรียน เทอม หน่วยกิตต่อภาค",
    "3.2 การดำเนินการหลักสูตร : ภาคการศึกษาต้น ภาคการศึกษาปลาย เปิดสอน เดือน ปีการศึกษา",
    "3.3 โครงสร้างหลักสูตร : หมวดวิชาศึกษาทั่วไป หมวดวิชาเฉพาะ หมวดวิชาเลือก หมวดวิชาเสรี รายวิชาบังคับ วิชาเลือก",
    "ตัวอย่างแผนการศึกษา : แผนการศึกษา, ปีที่ 1, ปีที่ 2, ปีที่ 3, ปีที่ 4, ภาคการศึกษาที่ 1, ภาคการศึกษาที่ 2,  หน่วยกิตสะสม",
    "คำอธิบายรายวิชา : คำอธิบายรายวิชา, เงื่อนไขของรายวิชา,** XX XXX XX ชื่อวิชาภาษาไทย ชื่อวิชาภาษาอังกฤษ เงื่อนไขของรายวิชา (if any)",
    "การฝึกงาน : ฝึกงาน, ไม่นับหน่วยกิต, ปีที่ 3"
    "สหกิจศึกษา : การเตรียมความพร้อมก่อนปฏิบัติงานสหกิจศึกษา, สหกิจศึกษา, 6 หน่วยกิต, ปีที่ 4"
    "3.4 ประสบการณ์ภาคสนาม : ฝึกงาน สหกิจ Intern Cooperative Education ปี 4 เตรียมความพร้อม Resume",
    "3.5 โครงงานหรืองานวิจัย : โครงงานวิทยาการคอมพิวเตอร์ Project I Project II วิจัย Research Methodology",
    "รหัสวิชา : ระบบรหัสรายวิชา",

    # หมวดที่ 5 การประเมินผล
    "เกณฑ์การสำเร็จการศึกษาตามหลักสูตร : เกณฑ์การสำเร็จการศึกษา, หลักสูตร, จำนวนหน่วยกิตที่ต้องสำเร็จ, การเลือกเรียนรายวิชา",

    # ภาคผนวก ข
    "ภาคผนวก ข ประวัติอาจารย์ประจำหลักสูตร : อาจารย์ ประวัติ ตำแหน่งทางวิชาการ การศึกษา ผลงานวิจัย งานวิชาการ การสอน ภาระงาน ระดับปริญญาตรี โท เอก",
    "คำค้นหาทั่วไป : หลักสูตร, รายวิชา, หน่วยกิต, การลงทะเบียน, แผนการศึกษา, ภาคการศึกษา, การฝึกงาน, สหกิจศึกษา, โครงงาน, อาชีพหลังเรียนจบ, คุณสมบัติผู้สำเร็จการศึกษา, ทักษะที่ได้รับ",

]


## baseline rag

In [ ]:
from langchain.embeddings.base import Embeddings
@dataclass
class BaselineCourseConfig:
    """Configuration for each course in baseline RAG"""
    name: str
    file_path: str
    elective_file_path: Optional[str] = None
    offset: int = 0
    chunk_size: int = 300  # 300 words per chunk
    chunk_overlap: int = 50  # 50 words overlap


class BaselineRAG:
    """
    Baseline RAG system that processes course documents with PyThaiNLP tokenization
    and chunking with 300 words and 50 overlap.
    """

    def __init__(self, embedding_model: str = "openai", enable_metadata: bool = True):
        self.courses: Dict[str, BaselineCourseConfig] = {}
        self.vectorstores: Dict[str, FAISS] = {}
        self.all_final_docs: List[Document] = []
        self.enable_metadata = enable_metadata

        # Initialize embedding model
        if embedding_model.lower() == "openai":
            self.embeddings = OpenAIEmbeddings(
                model="text-embedding-3-large",
                openai_api_key=OPENAI_API_KEY,
            )
        else:
            self.embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
                encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
            )

        # Initialize keyword extractor for metadata generation
        if self.enable_metadata:
            self.keyword_extractor = EmbeddingKeywordExtractor(use_local_model=False)
            print("Metadata extraction enabled")
        else:
            self.keyword_extractor = None
            print("Metadata extraction disabled")

    @staticmethod
    def _batched(iterable, n):
        it = iter(iterable)
        while True:
            batch = [x for _, x in zip(range(n), it)]
            if not batch:
                break
            yield batch

    def first_line_preview(self, text: str, max_len: int = 140) -> str:
        """Get first line preview"""
        for line in text.splitlines():
            s = line.strip()
            if s:
                return (s[:max_len] + "…") if len(s) > max_len else s
        return ""

    def add_course(self, config: BaselineCourseConfig):
        """
        Add a new course configuration to the RAG system.

        Args:
            config: BaselineCourseConfig instance
        """
        self.courses[config.name] = config
        print(f"Added course: {config.name}")

    def load_and_concatenate_documents(self, course_name: str) -> str:
        """
        Load and concatenate full document and elective document for a course.

        Args:
            course_name: Name of the course

        Returns:
            Concatenated document text
        """
        if course_name not in self.courses:
            raise ValueError(f"Course {course_name} not found in configuration")

        config = self.courses[course_name]

        # Load main document
        try:
            with open(config.file_path, 'r', encoding='utf-8') as f:
                main_doc = f.read()[config.offset:]
        except Exception as e:
            print(f"Error loading main document for {course_name}: {e}")
            main_doc = ""

        # Load elective document if available
        elective_doc = ""
        if config.elective_file_path and os.path.exists(config.elective_file_path):
            try:
                with open(config.elective_file_path, 'r', encoding='utf-8') as f:
                    elective_doc = f.read()
                print(f"Loaded elective document for {course_name}")
            except Exception as e:
                print(f"Error loading elective document for {course_name}: {e}")

        # Concatenate documents
        concatenated = main_doc
        if elective_doc:
            concatenated += "\n\n" + elective_doc

        return concatenated

    def tokenize_with_pythai_nlp(self, text: str) -> List[str]:
        return word_tokenize(text, engine="newmm", keep_whitespace=False)

    def create_chunks_with_overlap(self, tokens: List[str], chunk_size: int = 300, overlap: int = 50) -> List[List[str]]:
        if not tokens:
            return []

        chunks = []
        start = 0

        while start < len(tokens):
            end = min(start + chunk_size, len(tokens))
            chunks.append(tokens[start:end])

            # Move start position, accounting for overlap
            start = end - overlap

            # If we've reached the end, break
            if end >= len(tokens):
                break

        return chunks

    def process_course(self, course_name: str) -> List[Document]:

        print(f"\nProcessing course: {course_name}")
        config = self.courses[course_name]

        # Load and concatenate documents
        concatenated_text = self.load_and_concatenate_documents(course_name)

        if not concatenated_text.strip():
            print(f"No content found for course {course_name}")
            return []

        # Tokenize with PyThaiNLP
        tokens = self.tokenize_with_pythai_nlp(concatenated_text)
        print(f"Tokenized into {len(tokens)} tokens")

        # Create chunks with overlap
        token_chunks = self.create_chunks_with_overlap(
            tokens, config.chunk_size, config.chunk_overlap
        )
        print(f"Created {len(token_chunks)} chunks")

        # Convert chunks to Document objects with metadata extraction
        documents = []
        chunk_texts = [" ".join(chunk) for chunk in token_chunks]

        if self.enable_metadata and self.keyword_extractor:
            print("Extracting metadata for chunks...")

            # Batch extract keywords for all chunks
            BATCH_SIZE = 20   # ~50 chunks/request is a safe default for 300-word chunks
            all_keywords = []
            for batch in self._batched(chunk_texts, BATCH_SIZE):
                kws = self.keyword_extractor.extract_keywords_batch(batch, top_k=12)
                all_keywords.extend(kws)

            # Extract tags and other metadata for each chunk
            for i, (chunk_text, keywords) in enumerate(zip(chunk_texts, all_keywords)):
                # Get first line preview
                first_line = self.first_line_preview(chunk_text)

                # Extract header tags
                tags, tag_weights = self.keyword_extractor.header_tags_for_chunk(
                    chunk_text, top_k=3, tau=0.30
                )

                # Filter keywords by tags
                filtered_keywords = self.keyword_extractor.filter_keywords_by_tags(
                    keywords, tags, top_m=12, tau_kw=0.35
                )

                # Create enhanced metadata
                metadata = {
                    "course_name": course_name,
                    "chunk_id": i + 1,
                    "total_chunks": len(token_chunks),
                    "chunk_size": len(token_chunks[i]),
                    "chunking_method": "pythai_nlp_fixed_overlap",
                    # Enhanced metadata from keyword extraction
                    "layer": "baseline",
                    "part": i + 1,
                    "parts_total": len(token_chunks),
                    "first_line": first_line,
                    "keywords": filtered_keywords,
                    "tags": tags,
                    "tag_weights": tag_weights,
                }

                doc = Document(page_content=chunk_text, metadata=metadata)
                documents.append(doc)

                if i % 10 == 0:  # Progress indicator
                    print(f"Processed {i+1}/{len(token_chunks)} chunks with metadata")
        else:
            # Basic metadata without keyword extraction
            for i, chunk_text in enumerate(chunk_texts):
                metadata = {
                    "course_name": course_name,
                    "chunk_id": i + 1,
                    "total_chunks": len(token_chunks),
                    "chunk_size": len(token_chunks[i]),
                    "chunking_method": "pythai_nlp_fixed_overlap",
                    # Minimal metadata for compatibility
                    "layer": "baseline",
                    "part": i + 1,
                    "parts_total": len(token_chunks),
                    "first_line": chunk_text[:140] + "…" if len(chunk_text) > 140 else chunk_text,
                    "keywords": [],
                    "tags": [],
                    "tag_weights": {},
                }

                doc = Document(page_content=chunk_text, metadata=metadata)
                documents.append(doc)

        return documents

    def _build_faiss_batched(self, docs, *, init_batch_docs=12, min_batch_docs=2):
        """
        Build a FAISS store by embedding documents in small batches to avoid
        OpenAI's 300k-total-tokens-per-request limit.
        """
        if not docs:
            return None

        # Start with a conservative batch size; we'll shrink on 400 errors.
        batch_docs = max(init_batch_docs, min_batch_docs)

        # 1) bootstrap the index with the first batch
        start = 0
        while True:
            try:
                first_batch = docs[start:start+batch_docs]
                if not first_batch:
                    return None
                vs = FAISS.from_documents(first_batch, self.embeddings)
                start += batch_docs
                break
            except Exception as e:
                # If we hit the 300k cap or similar, reduce and retry
                msg = str(e).lower()
                if ("max_tokens_per_request" in msg or "300000" in msg or "badrequesterror" in msg) and batch_docs > min_batch_docs:
                    batch_docs = max(min_batch_docs, batch_docs // 2)
                    print(f"[faiss-init] shrinking batch to {batch_docs} due to token cap")
                else:
                    raise

        # 2) add the rest in batches, shrinking if we trip the cap
        while start < len(docs):
            try:
                end = min(len(docs), start + batch_docs)
                vs.add_documents(docs[start:end])
                start = end
            except Exception as e:
                msg = str(e).lower()
                if ("max_tokens_per_request" in msg or "300000" in msg or "badrequesterror" in msg) and batch_docs > min_batch_docs:
                    batch_docs = max(min_batch_docs, batch_docs // 2)
                    print(f"[faiss-add] shrinking batch to {batch_docs} due to token cap; retrying this segment")
                    # do not advance 'start'; just retry with smaller batch
                else:
                    raise
        return vs


    def build_vectorstores(self):
        """
        Build vector stores for all courses.
        """
        all_docs = []

        for course_name in self.courses:
            print(f"\nBuilding vectorstore for course: {course_name}")
            course_docs = self.process_course(course_name)

            if course_docs:
                # Create vector store for this course
                self.vectorstores[course_name] = self._build_faiss_batched(
                    course_docs,
                    init_batch_docs=12,   # safe default; adjust upward if you like
                    min_batch_docs=2
                )
                all_docs.extend(course_docs)
                print(f"Created vectorstore for {course_name} with {len(course_docs)} documents")
            else:
                print(f"No documents created for {course_name}")

        self.all_final_docs = all_docs
        print(f"\nBuilt vectorstores for {len(self.vectorstores)} courses with {len(all_docs)} total documents")

    def get_course_vectorstore(self, course_name: str) -> Optional[FAISS]:
        return self.vectorstores.get(course_name)

    def get_all_vectorstores(self) -> Dict[str, FAISS]:
        return self.vectorstores

    def save_vectorstores(self, output_dir: str = "vectorstores"):
        os.makedirs(output_dir, exist_ok=True)

        for course_name, vectorstore in self.vectorstores.items():
            save_path = os.path.join(output_dir, f"{course_name}_vectorstore")
            vectorstore.save_local(save_path)
            print(f"Saved vectorstore for {course_name} to {save_path}")

    def load_vectorstores(
        self,
        input_dir: str = "vectorstores",
        index_name: str = "index",
        *,
        populate_courses: bool = True,
        build_global: bool = False,
        rebuild_all_docs: bool = False,
    ):
        import os
        from langchain_community.vectorstores import FAISS
        from langchain.embeddings.base import Embeddings

        def is_vs_dir(p):
            return os.path.isfile(os.path.join(p, f"{index_name}.faiss")) and any(
                os.path.isfile(os.path.join(p, n)) for n in (f"{index_name}.pkl", "docstore.pkl", "index.pkl")
            )

        if not os.path.isdir(input_dir):
            print(f"[load] missing dir: {input_dir}")
            return {"loaded": 0, "courses": 0, "global_ntotal": 0, "docs": 0}

        # discover: root and 1-level subdirs
        candidates = []
        if is_vs_dir(input_dir): candidates.append(("default", input_dir))
        for d in os.listdir(input_dir):
            p = os.path.join(input_dir, d)
            if os.path.isdir(p) and is_vs_dir(p):
                cname = d.replace("_vectorstore", "").replace("_faiss", "").strip() or "default"
                candidates.append((cname, p))
        if not candidates:
            print(f"[load] no FAISS files under {input_dir}")
            return {"loaded": 0, "courses": 0, "global_ntotal": 0, "docs": 0}

        # load
        self.vectorstores = {}
        for cname, folder in candidates:
            try:
                vs = FAISS.load_local(folder, self.embeddings, index_name=index_name, allow_dangerous_deserialization=True)
                if not isinstance(getattr(vs, "embedding_function", None), Embeddings):
                    vs.embedding_function = self.embeddings
                self.vectorstores[cname] = vs
                ntotal = getattr(vs.index, "ntotal", 0)
                print(f"[load] {cname}: {ntotal} vectors")
            except Exception as e:
                print(f"[load] ERROR {folder}: {e}")

        # optional: ensure router has courses
        if populate_courses and not self.courses:
            for cname in self.vectorstores:
                self.courses[cname] = BaselineCourseConfig(name=cname, file_path="")

        # optional: build global
        global_ntotal = 0
        if build_global:
            self.global_vectorstore = None
            for vs in self.vectorstores.values():
                if self.global_vectorstore is None: self.global_vectorstore = vs
                else: self.global_vectorstore.merge_from(vs)
            if self.global_vectorstore:
                if not isinstance(getattr(self.global_vectorstore, "embedding_function", None), Embeddings):
                    self.global_vectorstore.embedding_function = self.embeddings
                global_ntotal = getattr(self.global_vectorstore.index, "ntotal", 0)
                print(f"[load] global: {global_ntotal} vectors")

        # optional: rebuild all_final_docs
        docs = 0
        if rebuild_all_docs:
            all_docs = []
            for cname, vs in self.vectorstores.items():
                ds = getattr(vs, "docstore", None)
                ids_map = getattr(vs, "index_to_docstore_id", None)
                if ds is None: continue
                if isinstance(ids_map, dict) and ids_map:
                    ids = [ids_map[i] for i in sorted(ids_map)]
                    fetch = lambda _id: getattr(ds, "search", lambda x: None)(_id) or getattr(ds, "_dict", {}).get(_id)
                    seq = filter(None, (fetch(_id) for _id in ids))
                else:
                    seq = getattr(ds, "_dict", {}).values() if hasattr(ds, "_dict") else []
                for doc in seq:
                    if "course_name" not in doc.metadata:
                        doc.metadata = dict(doc.metadata); doc.metadata["course_name"] = cname
                    all_docs.append(doc)
            self.all_final_docs = all_docs
            docs = len(all_docs)
            print(f"[load] docs rebuilt: {docs}")

        return {"loaded": len(self.vectorstores), "courses": len(self.courses), "global_ntotal": global_ntotal, "docs": docs}


## multiple rag

In [ ]:
from pathlib import Path
from typing import Union
import json
import shutil
from langchain.embeddings.base import Embeddings

class MultiDocumentRAG:
    """Main class for handling multiple หลักสูตร documents"""

    def __init__(self):
        self.processor = DocumentProcessor()
        self.keyworder = EmbeddingKeywordExtractor()
        self.all_final_docs = []
        self.courses: Dict[str, CourseConfig] = {}
        self.vectorstores: Dict[str, FAISS] = {}
        self.global_vectorstore: Optional[FAISS] = None
        self.strategies = {
            "semantic": SemanticSplitter(),
            "subtoc": SubTocSplitter(),
            "regex": RegexSplitter(),
        }
        self.embeddings: Embeddings = self.processor.embeddings

    def add_course(self, config: CourseConfig):
        """Add a new course configuration"""
        self.courses[config.name] = config

    def chunk_by_titles(self, text: str, titles: List[str]) -> List[Dict[str, str]]:
        """Split text by section titles"""
        positions = [(t, text.find(t)) for t in titles if text.find(t) != -1]
        missing = set(titles) - {t for t, _ in positions}
        for m in sorted(missing):
            print(f"Warning: title '{m}' not found.")

        positions.sort(key=lambda x: x[1])
        chunks = []
        for i, (title, start) in enumerate(positions):
            end = positions[i + 1][1] if i + 1 < len(positions) else len(text)
            chunks.append({"title": title, "content": text[start:end].strip()})
        return chunks

    def too_long(self, content: str, max_chars: int = 3000, max_words: int = 600) -> bool:
        """Check if content is too long"""
        toks = word_tokenize(content, engine="newmm", keep_whitespace=False)
        return len(content) > max_chars or len(toks) > max_words

    def first_line_preview(self, text: str, max_len: int = 140) -> str:
        """Get first line preview"""
        for line in text.splitlines():
            s = line.strip()
            if s:
                return (s[:max_len] + "…") if len(s) > max_len else s
        return ""

    def force_split(self, content: str, max_chars: int = 3000, max_words: int = 600) -> List[str]:
        """
        Force split content into chunks based on max_chars or max_words.
        This is used when semantic splitting fails to split the content.
        """

        # Try to split by words first (more semantically meaningful)
        words = word_tokenize(content, engine="newmm", keep_whitespace=False)
        chunks = []
        current_chunk = []
        current_length = 0

        for word in words:
            # Check if adding this word would exceed our limits
            potential_length = current_length + len(word) + (1 if current_chunk else 0)  # +1 for space

            if potential_length > max_chars or len(current_chunk) >= max_words:
                # Save current chunk and start a new one
                chunks.append(" ".join(current_chunk))
                current_chunk = [word]
                current_length = len(word)
            else:
                # Add word to current chunk
                current_chunk.append(word)
                current_length = potential_length

        if current_chunk:
            chunks.append(" ".join(current_chunk))

        return chunks

    def apply_semantic_tertiary(self, sub_doc: Dict[str, str], meta_base: Dict[str, Any]) -> List[Document]:
        """Optimized semantic splitting with post-pass 'force split' for overlong chunks."""
        content = sub_doc["content"]

        # Helper: flatten and force-split any overlong chunks
        def _normalize_docs(docs: List[Document]) -> List[Document]:
            normalized: List[Document] = []
            for d in docs:
                pc = d.page_content
                if self.too_long(pc):
                    # Force split this chunk and copy metadata
                    for chunk in self.force_split(pc):
                        if chunk.strip():
                            normalized.append(Document(page_content=chunk, metadata=dict(d.metadata)))
                else:
                    normalized.append(d)
            return normalized

        # Always create an initial docs list (so it's defined)
        docs: List[Document] = []

        if self.too_long(content):
            # First pass: semantic split
            docs = self.processor.sem_splitter.create_documents([content])
        else:
            # Wrap as a single doc so the logic is consistent later
            docs = [Document(page_content=content, metadata={})]

        # Post-pass normalization: flatten any still-too-long chunks
        docs = _normalize_docs(docs)
        total = len(docs)

        # Batch process all documents
        contents = [d.page_content for d in docs]
        first_lines = [self.first_line_preview(c) for c in contents]

        # Batch extract keywords and tags
        raw_keywords_list = self.keyworder.extract_keywords_batch(contents, top_k=24)

        tags_list, tag_weights_list = [], []
        for c in contents:
            tags, tag_weights = self.keyworder.header_tags_for_chunk(c, top_k=3, tau=0.30)
            tags_list.append(tags)
            tag_weights_list.append(tag_weights)

        keywords_list = []
        for raw_keywords, tags in zip(raw_keywords_list, tags_list):
            keywords = self.keyworder.filter_keywords_by_tags(raw_keywords, tags, top_m=12, tau_kw=0.35)
            keywords_list.append(keywords)

        # Create output documents
        out: List[Document] = []
        for i, (d, fl, keywords, tags, tag_weights) in enumerate(
            zip(docs, first_lines, keywords_list, tags_list, tag_weights_list), 1
        ):

            if "ระบบ รหัส วิชา ใช้ ตาม ระบบ ของ มหาวิทยาลัยขอนแก่น " in d.page_content:
                continue

            d.metadata.update({
                **meta_base,
                "layer": "semantic",
                "part": i,
                "parts_total": total,
                "first_line": fl,
                "keywords": keywords,
                "tags": tags,
                "tag_weights": tag_weights,
            })
            out.append(d)

        return out


    def route_and_split(self, course_name: str, toc_header: str, content: str) -> List[Dict[str, str]]:
        config = self.courses[course_name]
        chunking_plan = config.chunking_plan or {}

        cfg = chunking_plan.get(toc_header, chunking_plan.get("__default__", {"method": "semantic"}))
        method = cfg.get("method", "semantic")
        data = cfg.get("data", {}) or {}

        if method == "subtoc" and "markers" not in data:
            data["markers"] = config.subheader_map.get(toc_header, [])

        splitter = self.strategies.get(method, self.strategies["semantic"])
        parts = splitter.split(toc_header, content, **data)


        # ---- NEW: รองรับหลายสาขา
        catalog = getattr(config, "electives_catalog", None)
        electives_label = getattr(config, "electives_label", "วิชาเลือกเพิ่มเติม")

        if method == "subtoc" and catalog:
            # เลือก program แบบยืดหยุ่น: ใช้ config.program_name ก่อน ถ้าไม่มีให้เดา
            program_name = getattr(config, "program_name", None) or infer_program_from_text(content, catalog)
            print(program_name )
            if program_name:
                parts = merge_electives_into_chunks(
                    chunks=parts,
                    program_name=program_name,
                    catalog=catalog,
                    header_label_for_append=electives_label
                )


            # ถ้าเดาไม่ได้ ก็ข้ามการฝัง เพื่อไม่เพิ่ม noise

        return parts

    def process_course(self, course_name: str) -> List[Document]:
        """Process a single course document"""
        config = self.courses[course_name]

        # Load text
        text = Path(config.file_path).read_text(encoding="utf-8")[config.offset:]

        # Split by section titles
        all_sections = self.chunk_by_titles(text, config.section_titles)
        sections = [s for s in all_sections if s["title"] in config.selected_headers]
        for sec in sections:
            print(f"Processing section: {sec['title']}")

        final_docs: List[Document] = []
        chunking_plan = config.chunking_plan or {}

        for sec in sections:
            title = sec["title"]
            sub_parts = self.route_and_split(course_name, title, sec["content"])

            # Get base metadata
            plan_meta = chunking_plan.get(title, chunking_plan.get("__default__", {})).get("metadata", {})
            method = chunking_plan.get(title, chunking_plan.get("__default__", {"method": "semantic"}))["method"]

            for sp in sub_parts:
                meta_base = {
                    "course_name": course_name,  # Add course name to metadata
                    "toc_title": title,
                    "sub_header": sp["title"],
                    "method": method,
                    **plan_meta,
                }

                # Apply semantic splitting if too long
                if  "3.2 ชื่อ เลขประจ าตัวบัตรประชาชน ต าแหน่งและคุณวุฒิของอาจารย์" in sp["content"] :
                    continue
                docs = self.apply_semantic_tertiary(sp, meta_base)
                final_docs.extend(docs)

        return final_docs

    def build_vectorstores(self):
        """Build vectorstores for all courses"""
        all_docs = []

        for course_name in self.courses:
            print(f"Processing course: {course_name}")
            course_docs = self.process_course(course_name)


            # Create individual course vectorstore
            blocked = ("ค าอธิบายระบบรหัสวิชา", "คำอธิบายระบบรหัสวิชา")
            if course_docs:
                course_docs = [
                    d for d in course_docs
                    if all(b not in (d.metadata.get("sub_header") or "") for b in blocked)
                ]
                self.vectorstores[course_name] = FAISS.from_documents(course_docs, self.processor.embeddings)
                all_docs.extend(course_docs)

        self.all_final_docs = all_docs
        # Create global vectorstore
        # if all_docs:
        #     self.global_vectorstore = FAISS.from_documents(all_docs, self.processor.embeddings)

        print(f"Built vectorstores for {len(self.vectorstores)} courses with {len(all_docs)} total documents")

    def save_vectorstores(self, output_dir: str = "vectorstores"):
        os.makedirs(output_dir, exist_ok=True)

        for course_name, vectorstore in self.vectorstores.items():
            save_path = os.path.join(output_dir, f"{course_name}_vectorstore")
            vectorstore.save_local(save_path)
            print(f"Saved vectorstore for {course_name} to {save_path}")


    def load_vectorstores(
        self,
        input_dir: str = "vectorstores",
        index_name: str = "index",
        *,
        populate_courses: bool = True,
        build_global: bool = False,
        rebuild_all_docs: bool = False,
    ):
        import os
        from langchain_community.vectorstores import FAISS
        from langchain.embeddings.base import Embeddings

        def is_vs_dir(p):
            return os.path.isfile(os.path.join(p, f"{index_name}.faiss")) and any(
                os.path.isfile(os.path.join(p, n)) for n in (f"{index_name}.pkl", "docstore.pkl", "index.pkl")
            )

        if not os.path.isdir(input_dir):
            print(f"[load] missing dir: {input_dir}")
            return {"loaded": 0, "courses": 0, "global_ntotal": 0, "docs": 0}

        # discover: root and 1-level subdirs
        candidates = []
        if is_vs_dir(input_dir): candidates.append(("default", input_dir))
        for d in os.listdir(input_dir):
            p = os.path.join(input_dir, d)
            if os.path.isdir(p) and is_vs_dir(p):
                cname = d.replace("_vectorstore", "").replace("_faiss", "").strip() or "default"
                candidates.append((cname, p))
        if not candidates:
            print(f"[load] no FAISS files under {input_dir}")
            return {"loaded": 0, "courses": 0, "global_ntotal": 0, "docs": 0}

        # load
        self.vectorstores = {}
        for cname, folder in candidates:
            try:
                vs = FAISS.load_local(folder, self.embeddings, index_name=index_name, allow_dangerous_deserialization=True)
                if not isinstance(getattr(vs, "embedding_function", None), Embeddings):
                    vs.embedding_function = self.embeddings
                self.vectorstores[cname] = vs
                ntotal = getattr(vs.index, "ntotal", 0)
                print(f"[load] {cname}: {ntotal} vectors")
            except Exception as e:
                print(f"[load] ERROR {folder}: {e}")

        # optional: ensure router has courses
        if populate_courses and not self.courses:
            for cname in self.vectorstores:
                self.courses[cname] = BaselineCourseConfig(name=cname, file_path="")

        # optional: build global
        global_ntotal = 0
        if build_global:
            self.global_vectorstore = None
            for vs in self.vectorstores.values():
                if self.global_vectorstore is None: self.global_vectorstore = vs
                else: self.global_vectorstore.merge_from(vs)
            if self.global_vectorstore:
                if not isinstance(getattr(self.global_vectorstore, "embedding_function", None), Embeddings):
                    self.global_vectorstore.embedding_function = self.embeddings
                global_ntotal = getattr(self.global_vectorstore.index, "ntotal", 0)
                print(f"[load] global: {global_ntotal} vectors")

        # optional: rebuild all_final_docs
        docs = 0
        if rebuild_all_docs:
            all_docs = []
            for cname, vs in self.vectorstores.items():
                ds = getattr(vs, "docstore", None)
                ids_map = getattr(vs, "index_to_docstore_id", None)
                if ds is None: continue
                if isinstance(ids_map, dict) and ids_map:
                    ids = [ids_map[i] for i in sorted(ids_map)]
                    fetch = lambda _id: getattr(ds, "search", lambda x: None)(_id) or getattr(ds, "_dict", {}).get(_id)
                    seq = filter(None, (fetch(_id) for _id in ids))
                else:
                    seq = getattr(ds, "_dict", {}).values() if hasattr(ds, "_dict") else []
                for doc in seq:
                    if "course_name" not in doc.metadata:
                        doc.metadata = dict(doc.metadata); doc.metadata["course_name"] = cname
                    all_docs.append(doc)
            self.all_final_docs = all_docs
            docs = len(all_docs)
            print(f"[load] docs rebuilt: {docs}")

        return {"loaded": len(self.vectorstores), "courses": len(self.courses), "global_ntotal": global_ntotal, "docs": docs}


## baseline query

In [ ]:
import json
from typing import List, Dict, Union, Tuple

class BaselineQuery:
    """
    Baseline query class that uses user question directly for query (no LLM rewrite)
    and applies equal weight distribution for each course.
    """

    def __init__(self):
        """Initialize the BaselineQuery class."""
        pass

    def rewrite_and_distribute(self, question: str, headers_text: str, course_list: List[str]) -> Tuple[str, List[Dict[str, Union[str, float]]]]:
        """
        Returns the original question as search query and equal distribution across courses.

        Args:
            question: The user's original question
            headers_text: Document headers (not used in baseline)
            course_list: List of available courses

        Returns:
            Tuple of (search_query, distributions)
            - search_query: The original question (unchanged)
            - distributions: List of dicts with equal weights for all courses
        """
        # Use the original question directly without any rewriting
        search_query = question

        # Create equal weight distribution for all courses
        if not course_list:
            return search_query, []

        # Calculate equal weight for each course
        equal_weight = 1.0 / len(course_list)

        # Create distribution list with equal weights
        distributions = [
            {"course": course_name, "weight": equal_weight}
            for course_name in course_list
        ]

        print(f"Baseline Query - Original: {question}")
        print(f"Baseline Query - Equal distribution across {len(course_list)} courses: {equal_weight:.3f} each")

        return search_query, distributions

    def get_default_headers_text(self) -> str:
        """
        Returns default headers text (not used in baseline but included for interface compatibility).
        """
        return """
        หมวดที่ 1 ข้อมูลทั่วไป
        หมวดที่ 3 ระบบการจัดการศึกษา การดำเนินการ และโครงสร้างหลักสูตร
        คำอธิบายรายวิชา
        แผนการศึกษา
        การฝึกงานและสหกิจศึกษา
        โครงงานหรืองานวิจัย
        เกณฑ์การสำเร็จการศึกษาตามหลักสูตร
        """


## query rewrite

In [ ]:
class QueryRewriter:
    """Handles query rewriting and analysis in a single LLM call"""

    def __init__(self, llm=None, **llm_kwargs):
        self.llm = llm or self._initialize_default_llm(**llm_kwargs)
        self.combined_prompt = self._create_combined_prompt()

    def _initialize_default_llm(self, **kwargs):
        """Initialize default LLM """
        return ChatOpenAI(
            model=kwargs.get("model", "deepseek/deepseek-chat-v3-0324"),
            temperature=kwargs.get("temperature", 0.1),
            openai_api_key=kwargs.get("openai_api_key"),  # pass via env/secret manager
            base_url=kwargs.get("base_url", "https://openrouter.ai/api/v1"),
        )

    def _create_combined_prompt(self):
        """Create a single prompt that returns both the Thai search query and course weights."""
        return ChatPromptTemplate.from_messages([
            ("system", """You are an expert search-query generator and course-relevance analyst.

Return a single JSON object with two fields:
1) "search_query": a concise Thai search Keyword query for retrieving content relevant to the user's question using the provided headers.
2) "distributions": a list of objects with "course" and "weight" for how relevant each course is to the user's intent.

STRICT RULES:
- Analyze the user's query for intent and topical focus.
- No comments,No any markdown, No trailing text outside JSON.
- Output JSON ONLY (no extra text).
"search_query":
    - Treat "search_query" as a query to find a doc only, NOT as a question.
    - Aim for maximum relevance; include adjacent/related sections if helpful.
    - Prefer terms that match the document structure and headings.
    - All queries MUST be Thai Keyword only.
    - In "search_query" Never include or infer a course name.
    - Queries should only contain text helpful for searching at the Document Level (in เอกสาร มคอ. only), Not course level.
    - Consider the provided document headers to align with structure/terminology.
"distributions":
    - Always assign 1.0 for the major that user mentioned, 0.0 for others.
    - If no major is detected → assign equal weights.
    - Weights must sum exactly to 1.0.
    - Keep course names exactly as provided in "distributions".
    - Sort "distributions" by weight descending.

Schema:
{{
"search_query": "…",
"distributions": [{{"course": "Course Name", "weight": 0.x}}, ...]
}}"""),
        ("user", """Question: {question}

Available document headers:
{headers}

Available courses (comma-separated):
{courses}""")
        ])


    def rewrite_and_distribute(self,question: str,headers_text: str,course_list: List[str]) -> Tuple[str, List[Dict[str, Union[str, float]]]]:
        """
        Single LLM call that returns (thai_search_query, course_distributions).
        """
        try:
            chain = self.combined_prompt | self.llm | StrOutputParser()
            raw = chain.invoke({
                "question": question,
                "headers": headers_text,
                "courses": ", ".join(course_list)
            }).strip()

            # Defensive cleanup for any accidental markup
            cleaned = re.sub(r"<[^>]+>", "", raw)

            data = json.loads(cleaned)
            print(data,"test================")

            # Extract and sanitize the Thai search query
            thai_query = (data.get("search_query") or "").strip()
            thai_query = re.sub(r"<[^>]+>", "", thai_query)

            # Extract distributions; fall back to equal spread if missing/invalid
            dists = data.get("distributions") or []
            # Ensure only known courses are present; if model invents, filter them out
            name_set = set(course_list)
            dists = [d for d in dists if isinstance(d, dict) and d.get("course") in name_set]

            # If empty or missing any courses, fill in zeroes so we can normalize
            if not dists:
                dists = [{"course": c, "weight": 1.0 / max(1, len(course_list))} for c in course_list]
            else:
                # Add zero-weight entries for courses the model omitted (optional)
                present = {d["course"] for d in dists}
                for c in course_list:
                    if c not in present:
                        dists.append({"course": c, "weight": 0.0})

            # Normalize weights to sum to 1.0 (robust to rounding/model drift)
            total = sum(float(d.get("weight", 0.0)) for d in dists)
            if total <= 0:
                w = 1.0 / len(course_list)
                dists = [{"course": c, "weight": w} for c in course_list]
            else:
                dists = [{"course": d["course"], "weight": float(d.get("weight", 0.0)) / total} for d in dists]

            # Optional: round for display stability, then re-normalize to ensure exact 1.0
            dists = [{"course": d["course"], "weight": round(d["weight"], 6)} for d in dists]
            total = sum(d["weight"] for d in dists)
            if total != 1.0 and total > 0:
                # adjust the largest weight to fix rounding drift
                idx_max = max(range(len(dists)), key=lambda i: dists[i]["weight"])
                dists[idx_max]["weight"] = round(dists[idx_max]["weight"] + (1.0 - total), 6)

            # Sort descending by weight as required
            dists.sort(key=lambda x: x["weight"], reverse=True)

            return thai_query, dists

        except Exception as e:
            # Fallbacks: return original question as query (Thai might be required upstream)
            # and equal distribution across provided courses.
            print(f"Combined analyze failed: {e} ====")
            fallback_query = question
            w = 1.0 / max(1, len(course_list))
            fallback_dists = [{"course": c, "weight": w} for c in course_list]
            return fallback_query, fallback_dists

    def get_default_headers_text(self) -> str:
        """Get default headers text for Thai curriculum documents"""
        return "\n".join([
    # หมวดที่ 1 ข้อมูลทั่วไป
    "1. รหัสและชื่อหลักสูตร : หลักสูตร ชื่อหลักสูตร ชื่อปริญญา ชื่อเต็ม ชื่อย่อ ไทย อังกฤษ",
    "2. ชื่อปริญญาและสาขาวิชา",
    "4. จำนวนหน่วยกิตที่เรียนตลอดหลักสูตร : หน่วยกิต รวมทั้งหมด",
    "5. รูปแบบของหลักสูตร : หลักสูตร 4 ปี ปริญญาตรี ภาษาที่ใช้ การรับเข้า นักศึกษาไทย ต่างชาติ",

    # หมวดที่ 3 ระบบการจัดการศึกษา การดำเนินการ และโครงสร้างหลักสูตร
    "ระบบการจัดการศึกษา : ระบบทวิภาค ภาคต้น ภาคปลาย ภาคเรียน เทอม หน่วยกิตต่อภาค",
    "การดำเนินการหลักสูตร : ภาคการศึกษาต้น ภาคการศึกษาปลาย เปิดสอน เดือน ปีการศึกษา",
    "โครงสร้างหลักสูตร : หมวดวิชาศึกษาทั่วไป หมวดวิชาเฉพาะ หมวดวิชาเลือก หมวดวิชาเสรี รายวิชาบังคับ วิชาเลือก",
    "ตัวอย่างแผนการศึกษา : [แผนการศึกษา AND ปีที่ X  AND ภาคการศึกษาที่ Y  AND รวมจ านวนหน่วยกิตลงทะเบียนเรียน],",
    "คำอธิบายรายวิชา : คำอธิบายรายวิชา, เงื่อนไขของรายวิชา,** XX XXX XX ชื่อวิชาภาษาไทย ชื่อวิชาภาษาอังกฤษ เงื่อนไขของรายวิชา (if any)",
    "การฝึกงาน : ฝึกงาน, ไม่นับหน่วยกิต, ปีที่ 3"
    "สหกิจศึกษา : การเตรียมความพร้อมก่อนปฏิบัติงานสหกิจศึกษา, สหกิจศึกษา, 6 หน่วยกิต, ปีที่ 4"
    "ประสบการณ์ภาคสนาม : ฝึกงาน สหกิจ Intern Cooperative Education ปี 4 เตรียมความพร้อม Resume",
    "โครงงานหรืองานวิจัย : โครงงาน Project I Project II วิจัย Research Methodology",
    "รหัสวิชา : ระบบรหัสรายวิชา",

    # หมวดที่ 5 การประเมินผล
    "เกณฑ์การสำเร็จการศึกษาตามหลักสูตร : เกณฑ์การสำเร็จการศึกษา, หลักสูตร, จำนวนหน่วยกิตที่ต้องสำเร็จ, การเลือกเรียนรายวิชา",

    # ภาคผนวก ข
    "ภาคผนวก ข ประวัติอาจารย์ประจำหลักสูตร : อาจารย์ ประวัติ ตำแหน่งทางวิชาการ การศึกษา ผลงานวิจัย งานวิชาการ การสอน ภาระงาน ระดับปริญญาตรี โท เอก",
    "คำค้นหาทั่วไป : หลักสูตร, รายวิชา, หน่วยกิต, การลงทะเบียน, แผนการศึกษา, ภาคการศึกษา, การฝึกงาน, สหกิจศึกษา, โครงงาน, อาชีพหลังเรียนจบ, คุณสมบัติผู้สำเร็จการศึกษา, ทักษะที่ได้รับ",

])

## multi query rewrite

In [ ]:
import json
import re
from typing import List, Dict, Union, Tuple
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser

class MultiQueryRewriter:
    """Handles query rewriting and analysis in a single LLM call"""

    def __init__(self, llm=None, max_atomic_queries=5, **llm_kwargs):
        self.llm = llm or self._initialize_default_llm(**llm_kwargs)
        self.max_atomic_queries = max_atomic_queries
        self.combined_prompt = self._create_combined_prompt()

    def _initialize_default_llm(self, **kwargs):
        """Initialize default LLM """
        return ChatOpenAI(
            model=kwargs.get("model", "deepseek/deepseek-chat-v3-0324"),
            temperature=kwargs.get("temperature", 0.1),
            openai_api_key=kwargs.get("openai_api_key"),  # pass via env/secret manager
            base_url=kwargs.get("base_url", "https://openrouter.ai/api/v1"),
        )

    def _create_combined_prompt(self):
        """Create a single prompt that returns atomic queries with course distributions."""
        return ChatPromptTemplate.from_messages([
        ("system", """You are an expert search-query generator and course-relevance analyst.

Return a single JSON object with two fields:
1) "atomic_queries": a list of concise Keyword search queries.
2) "distributions": a list of objects with "course" and "weight" for how relevant each course is to the user's intent.

STRICT RULES:
- Analyze the user's query for intent and topical focus.
- Break down the user's question into 1-3 atomic queries that together cover all aspects of the question.
- Course name should appear on "distributions" only, Not allow in other field.
- No comments,No any extra markdown, No trailing text outside JSON.
- Only format strict follow Schema only.
- Output JSON ONLY (no extra text).
"atomic_queries":
    - Each atomic query should be a query to find a specific Topic of document content, NOT a question.
    - Use headers as a guildline to Analyze to create queries
    - Each query targets ONE specific aspect/topic from the user's intent.
    - Aim for maximum relevance; include adjacent/related sections if helpful..
    - Queries must be search Thai terms/keywords, NOT questions or complete sentences.
    - In "search_query" Never include or infer a course name. Not needed, because it's no longer necessary to use it.
    - If a Subject name is in Thai, translate Subject name(only) into Subject name English ver.
    - Try to avoid duplicate or near-duplicate queries; If possible.
    - The number of atomic queries is flexible; it's can less than three.
    - Consider the provided document headers to align with structure/terminology.
"distributions":
    - Always assign 1.0 for the major that user mention, 0.0 for others.
    - If no major is mention, assign equal weights.
    - Weights must sum exactly to 1.0.
    - Keep course names exactly as provided in "distributions".
    - Sort "distributions" by weight descending.
Schema:
{{
"atomic_queries": ["query1", "query2", ...],
"distributions": [{{"course": "Course Name", "weight": 0.x}}, ...]
}}"""),
        ("user", """Question: {question}

Available document headers:
{headers}

Available courses (comma-separated):
{courses}""")
    ])

    def rewrite_and_distribute(self, question: str, headers_text: str, course_list: List[str]) -> Tuple[List[str], List[Dict[str, Union[str, float]]]]:
        """
        Single LLM call that returns (atomic_queries, course_distributions).
        """
        try:
            chain = self.combined_prompt | self.llm | StrOutputParser()
            raw = chain.invoke({
                "question": question,
                "headers": headers_text,
                "courses": ", ".join(course_list)
            }).strip()

            # Defensive cleanup for any accidental markup
            cleaned = re.sub(r"<[^>]+>", "", raw)
            data = json.loads(cleaned)

            # Extract and sanitize the atomic queries
            atomic_queries = data.get("atomic_queries", [])
            if not isinstance(atomic_queries, list):
                atomic_queries = [str(atomic_queries)]

            # Clean each query
            atomic_queries = [re.sub(r"<[^>]+>", "", str(query)).strip() for query in atomic_queries]

            # Filter out empty queries
            atomic_queries = [query for query in atomic_queries if query]

            # Ensure we don't exceed the maximum number of queries
            if len(atomic_queries) > self.max_atomic_queries:
                atomic_queries = atomic_queries[:self.max_atomic_queries]

            # If no valid queries, use the original question as a fallback
            if not atomic_queries:
                atomic_queries = [question]

            # Extract distributions; fall back to equal spread if missing/invalid
            dists = data.get("distributions") or []
            # Ensure only known courses are present; if model invents, filter them out
            name_set = set(course_list)
            dists = [d for d in dists if isinstance(d, dict) and d.get("course") in name_set]

            # If empty or missing any courses, fill in zeroes so we can normalize
            if not dists:
                dists = [{"course": c, "weight": 1.0 / max(1, len(course_list))} for c in course_list]
            else:
                # Add zero-weight entries for courses the model omitted (optional)
                present = {d["course"] for d in dists}
                for c in course_list:
                    if c not in present:
                        dists.append({"course": c, "weight": 0.0})

            # Normalize weights to sum to 1.0 (robust to rounding/model drift)
            total = sum(float(d.get("weight", 0.0)) for d in dists)
            if total <= 0:
                w = 1.0 / len(course_list)
                dists = [{"course": c, "weight": w} for c in course_list]
            else:
                dists = [{"course": d["course"], "weight": float(d.get("weight", 0.0)) / total} for d in dists]

            # Optional: round for display stability, then re-normalize to ensure exact 1.0
            dists = [{"course": d["course"], "weight": round(d["weight"], 6)} for d in dists]
            total = sum(d["weight"] for d in dists)
            if total != 1.0 and total > 0:
                # adjust the largest weight to fix rounding drift
                idx_max = max(range(len(dists)), key=lambda i: dists[i]["weight"])
                dists[idx_max]["weight"] = round(dists[idx_max]["weight"] + (1.0 - total), 6)

            # Sort descending by weight as required
            dists.sort(key=lambda x: x["weight"], reverse=True)

            return atomic_queries, dists

        except Exception as e:
            # Fallbacks: return original question as the only query
            # and equal distribution across provided courses.
            print(f"Combined analyze failed: {e}")
            fallback_queries = [question]
            w = 1.0 / max(1, len(course_list))
            fallback_dists = [{"course": c, "weight": w} for c in course_list]
            return fallback_queries, fallback_dists

    def get_default_headers_text(self) -> str:
        """Get default headers text for Thai curriculum documents"""
        return "\n".join([
    # หมวดที่ 1 ข้อมูลทั่วไป
    "1. รหัสและชื่อหลักสูตร : หลักสูตร ชื่อหลักสูตร ชื่อปริญญา ชื่อเต็ม ชื่อย่อ ไทย อังกฤษ",
    "2. ชื่อปริญญาและสาขาวิชา",
    "4. จำนวนหน่วยกิตที่เรียนตลอดหลักสูตร : หน่วยกิต รวมทั้งหมด",
    "5. รูปแบบของหลักสูตร : หลักสูตร 4 ปี ปริญญาตรี ภาษาที่ใช้ การรับเข้า นักศึกษาไทย ต่างชาติ",

    # หมวดที่ 3 ระบบการจัดการศึกษา การดำเนินการ และโครงสร้างหลักสูตร
    "ระบบการจัดการศึกษา : ระบบทวิภาค ภาคต้น ภาคปลาย ภาคเรียน เทอม หน่วยกิตต่อภาค",
    "การดำเนินการหลักสูตร : ภาคการศึกษาต้น ภาคการศึกษาปลาย เดือน ปีการศึกษา",
    "โครงสร้างหลักสูตร : หมวดวิชาศึกษาทั่วไป หมวดวิชาเฉพาะ หมวดวิชาเลือก หมวดวิชาเสรี รายวิชาบังคับ วิชาเลือก",
    "ตัวอย่างแผนการศึกษา : แผนการศึกษา, ปีที่ x, ภาคการศึกษาที่ y,  วิชาเลือกที่ลงได้,รวมจ านวนหน่วยกิตลงทะเบียนเรียน",
    "คำอธิบายรายวิชา : [คำอธิบายรายวิชา:main key], เงื่อนไขของรายวิชา: [pre-req sub], คำอธิบายรายวิชา XX XXX XX  [subject name here] เงื่อนไขของรายวิชา",
    "การฝึกงาน : ฝึกงาน, ไม่นับหน่วยกิต, ปีที่ 3",
    "สหกิจศึกษา : การเตรียมความพร้อมก่อนปฏิบัติงานสหกิจศึกษา, สหกิจศึกษา, 6 หน่วยกิต, ปีที่ 4",
    "ประสบการณ์ภาคสนาม : ฝึกงาน สหกิจ Intern Cooperative Education ปี 4 เตรียมความพร้อม Resume",
    "โครงงานหรืองานวิจัย : โครงงาน Project I Project II วิจัย Research Methodology",
    "รหัสวิชา : ระบบรหัสรายวิชา",

    # หมวดที่ 5 การประเมินผล
    "เกณฑ์การสำเร็จการศึกษาตามหลักสูตร : เกณฑ์การสำเร็จการศึกษา, หลักสูตร, จำนวนหน่วยกิตที่ต้องสำเร็จ, การเลือกเรียนรายวิชา",

    "คำค้นหาทั่วไป : หลักสูตร, รายวิชา, หน่วยกิต, การลงทะเบียน, แผนการศึกษา, ภาคการศึกษา, การฝึกงาน, สหกิจศึกษา, โครงงาน, อาชีพหลังเรียนจบ, คุณสมบัติผู้สำเร็จการศึกษา, ทักษะที่ได้รับ",
])

## baseline retriever

### baseline

In [ ]:
class BaselineRetriever(BaseRetriever):
    """
    Simple baseline retriever that performs vector search without any reranking
    or complex scoring. Just returns top-k results based on vector similarity.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 rag_system,
                 distributed_searcher,
                 query_rewriter=None,
                 **kwargs):

        super().__init__(**kwargs)
        self.rag_system = rag_system
        self.distributed_searcher = distributed_searcher
        self.K = kwargs.get('K', 20)  # Total docs to retrieve
        self.k = kwargs.get('k', 4)   # Final docs to return
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)

        # Use baseline query rewriter if none provided
        if query_rewriter is None:
            self.query_rewriter = BaselineQuery()
        else:
            self.query_rewriter = query_rewriter

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """
        Retrieve documents using simple vector search without reranking.

        Args:
            query: The search query

        Returns:
            List of documents sorted by vector similarity
        """
        search_query = query
        distributions = None

        # Use baseline query rewriting (which just returns original query + equal distribution)
        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_query, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Baseline - Using query: {search_query}")
            print(f"Baseline - Distribution: {distributions}")

        # Perform distributed vector search
        results_with_scores = self.distributed_searcher.search_with_scores(
            search_query, self.K, distributions
        )

        if not results_with_scores:
            print("No documents found in vector search")
            return []

        # Extract documents and sort by vector similarity score (lower distance = better)
        # Convert distance to similarity for intuitive understanding
        documents_with_similarity = []
        for doc, vector_distance, weight in results_with_scores:
            # Convert distance to similarity (lower distance = higher similarity)
            similarity_score = 1.0 / (1.0 + vector_distance)
            documents_with_similarity.append((doc, similarity_score, weight))

        # Sort by similarity score (descending)
        documents_with_similarity.sort(key=lambda x: x[1], reverse=True)

        # Return top-k documents
        top_documents = [doc for doc, _, _ in documents_with_similarity[:self.k]]

        print(f"\n🔍 Baseline Retriever - Returning top {len(top_documents)} documents by vector similarity:")
        for i, doc in enumerate(top_documents, 1):
            metadata = doc.metadata or {}
            course_name = metadata.get('course_name', 'Unknown')
            toc_title = metadata.get('toc_title', 'Unknown')
            similarity = documents_with_similarity[i-1][1]
            print(f"#{i}: [{course_name}] {toc_title} (similarity: {similarity:.4f})")

        return top_documents

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        """Async version of _get_relevant_documents"""
        return self._get_relevant_documents(query)

### multi baseline

In [ ]:

class MultiBaselineRetriever(BaseRetriever):
    """
    Multi-query version of baseline retriever that handles multiple atomic queries
    but still uses simple vector search without reranking.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 rag_system,
                 distributed_searcher,
                 query_rewriter=None,
                 max_queries=5,
                 **kwargs):
        """
        Initialize the multi-query baseline retriever.

        Args:
            rag_system: The RAG system containing vectorstores
            distributed_searcher: Component for searching across multiple courses
            query_rewriter: Optional query rewriter (defaults to MultiBaselineQuery if None)
            max_queries: Maximum number of atomic queries to process
            **kwargs: Additional parameters including K and k
        """
        super().__init__(**kwargs)
        self.rag_system = rag_system
        self.distributed_searcher = distributed_searcher
        self.K = kwargs.get('K', 20)  # Total docs to retrieve per query
        self.k = kwargs.get('k', 4)   # Final docs to return per query
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)
        self.max_queries = max_queries

        # Use multi baseline query rewriter if none provided
        self.query_rewriter = query_rewriter

    def _get_relevant_documents(self, query: str) -> List[Document]:

        search_queries = [query]  # Default to single query
        distributions = None

        # Use multi-query baseline rewriting
        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_queries, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Multi Baseline - Atomic queries: {search_queries}")
            print(f"Multi Baseline - Distribution: {distributions}")

            # Limit number of queries
            if len(search_queries) > self.max_queries:
                print(f"Limiting queries from {len(search_queries)} to {self.max_queries}")
                search_queries = search_queries[:self.max_queries]

        # If only one query, use simple baseline retriever logic
        if len(search_queries) == 1:
            return self._search_single_query(search_queries[0], distributions)

        # Process multiple queries and collect unique results
        all_results = []
        seen_docs = set()

        for i, search_query in enumerate(search_queries):
            print(f"\n--- Processing Query {i+1}/{len(search_queries)}: '{search_query}' ---")

            query_docs = self._search_single_query(search_query, distributions)

            # Add unique documents
            for doc in query_docs:
                doc_id = str(hash(doc.page_content))
                if doc_id not in seen_docs:
                    seen_docs.add(doc_id)
                    all_results.append(doc)

            print(f"Retrieved {len(query_docs)} documents for query {i+1}")

        print(f"\nMulti Baseline - Returning {len(all_results)} unique documents from {len(search_queries)} queries")
        for i, doc in enumerate(all_results, 1):
            metadata = doc.metadata or {}
            course_name = metadata.get('course_name', 'Unknown')
            toc_title = metadata.get('toc_title', 'Unknown')
            print(f"#{i}: [{course_name}] {toc_title}")

        return all_results

    def _search_single_query(self, search_query: str, distributions: Optional[List[Dict[str, Any]]]) -> List[Document]:

        # Perform distributed vector search
        results_with_scores = self.distributed_searcher.search_with_scores(
            search_query, self.K, distributions
        )

        if not results_with_scores:
            return []

        # Convert distance to similarity and sort
        documents_with_similarity = []
        for doc, vector_distance, weight in results_with_scores:
            similarity_score = 1.0 / (1.0 + vector_distance)
            documents_with_similarity.append((doc, similarity_score, weight))

        documents_with_similarity.sort(key=lambda x: x[1], reverse=True)

        return [doc for doc, _, _ in documents_with_similarity[:self.k]]

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        """Async version of _get_relevant_documents"""
        return self._get_relevant_documents(query)

## retrival helper

### helper

In [ ]:
class TextProcessingUtils:
    @staticmethod
    def normalize_text(s: str) -> str:
        import unicodedata, re as _re
        if not s:
            return ""
        THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
        s = unicodedata.normalize("NFKC", s).translate(THAI_DIGITS).lower()
        return _re.sub(r"\s+", " ", s).strip()

    @staticmethod
    def extract_keywords(q: str):
        import re as _re
        from pythainlp.corpus.common import thai_stopwords
        from pythainlp import word_tokenize
        qn = TextProcessingUtils.normalize_text(q)
        toks = word_tokenize(qn, engine="newmm")
        stopwords = set(thai_stopwords())
        toks = [t for t in toks if _re.fullmatch(r"[0-9]+|[a-zA-Zก-๙]+", t)]
        return [t for t in toks if t not in stopwords]

    @staticmethod
    def minmax_normalize(xs):
        """Return list normalized to [0,1]; if flat, returns 0.5 for all."""
        if not xs:
            return []
        lo, hi = float(min(xs)), float(max(xs))
        if hi - lo < 1e-9:
            return [0.5] * len(xs)
        return [(x - lo) / (hi - lo) for x in xs]


### distibute search

In [ ]:
from typing import List, Tuple, Dict, Any, Optional
from langchain.schema import Document
import math

class DistributedSearcher:
    """Distributed retrieval across multiple per-course vectorstores."""

    def __init__(self, rag_system, K: int = 20):
        self.rag_system = rag_system
        self.K = K  # default total_k

    def _available_courses(self) -> List[str]:
        if getattr(self.rag_system, "courses", None):
            return [c for c in self.rag_system.courses.keys()
                    if c in getattr(self.rag_system, "vectorstores", {})]
        return list(getattr(self.rag_system, "vectorstores", {}).keys())

    def _minmax_normalize(self, scores: List[float]) -> List[float]:
        if not scores:
            return []
        mn, mx = min(scores), max(scores)
        if mx - mn < 1e-12:
            return [0.0 for _ in scores]
        return [(s - mn) / (mx - mn) for s in scores]

    def search_with_scores(
        self,
        query: str,
        total_k: Optional[int] = None,
        distributions: Optional[List[Dict[str, float]]] = None,
    ) -> List[Tuple[Document, float, float]]:
        """Search all vectorstores and merge results by score."""
        total_k = total_k or self.K
        courses = self._available_courses()
        if not courses:
            print("[DistributedSearcher] No vectorstores found.")
            return []

        # Compute weights per course
        if not distributions:
            w = 1.0 / len(courses)
            distributions = [{"course": c, "weight": w} for c in courses]

        wsum = sum(d.get("weight", 0.0) for d in distributions)
        if wsum != 0.0:
            for d in distributions:
                d["weight"] = d.get("weight", 0.0) / wsum
        else:
            # All weights are zero; per your requirement, skip retrieval entirely.
            print("[DistributedSearcher] All weights are zero; skipping retrieval.")
            return []

        # Retrieve per course (skip zero-weight buckets)
        all_results: List[Tuple[Document, float, float, float]] = []
        for dist in distributions:
            cname = dist["course"]
            weight = dist["weight"]

            # >>> key change: skip retrieval when weight <= 0
            if weight <= 0.0:
                print(f"[DistributedSearcher] Skipping {cname} (w=0)")
                continue

            # allow k_for_course to be zero naturally; skip if zero
            k_for_course = int(total_k * weight)
            if k_for_course <= 0:
                print(f"[DistributedSearcher] Skipping {cname} (k=0 from weight={weight:.4f})")
                continue

            vs = self.rag_system.vectorstores.get(cname)
            if not vs:
                continue
            try:
                results = vs.similarity_search_with_score(query, k=k_for_course)
                scores = [s for _, s in results]
                norm = self._minmax_normalize(scores)
                for (doc, raw), n in zip(results, norm):
                    if "course_name" not in doc.metadata:
                        doc.metadata = dict(doc.metadata)
                        doc.metadata["course_name"] = cname
                    all_results.append((doc, raw, weight, n))
                print(f"[DistributedSearcher] Retrieved {len(results)} from {cname} (w={weight:.2f})")
            except Exception as e:
                print(f"[DistributedSearcher] Search failed for {cname}: {e}")

        if not all_results:
            return []

        # Combine and rank
        combined = []
        for doc, raw, w, nrm in all_results:
            # w > 0 always here due to skipping zero-weight buckets
            combined_score = nrm / w
            combined.append((combined_score, doc, raw, w))
        combined.sort(key=lambda x: (x[0], x[2]))

        # Return top total_k
        return [(doc, raw, w) for (_, doc, raw, w) in combined[:total_k]]


### metadata score

In [ ]:
class MetadataScore:
    def __init__(self,
                 first_line_boost=0.15,
                 sub_header_boost=0.08,
                 toc_title_boost=0.1,
                 keywords_boost=0.25,
                 tag_unit_bonus=0.5,
                 layer_bias=None,
                 metadata_total_cap=0.45):
        self.first_line_boost = first_line_boost
        self.sub_header_boost = sub_header_boost
        self.toc_title_boost = toc_title_boost
        self.keywords_boost = keywords_boost
        self.tag_unit_bonus = tag_unit_bonus
        self.layer_bias = layer_bias or {"semantic": 0.04, "second": 0.02}
        self.metadata_total_cap = metadata_total_cap

    def kw_hits(self, kws_set, text: str) -> int:
        if not text:
            return 0
        toks = set(TextProcessingUtils.extract_keywords(text))
        return len(kws_set & toks)

    def overlap_hits(self, query_terms: set, candidates) -> int:
        """Count exact hits between query terms and a list/string of candidates."""
        if not candidates:
            return 0
        if isinstance(candidates, str):
            items = [c.strip() for c in candidates.replace(",", " ").split() if c.strip()]
        else:
            items = [str(c).strip() for c in candidates if str(c).strip()]
        cand_set = set(items)
        return sum(1 for t in query_terms if t in cand_set)

    def soft_gain(self, hits: int, unit: float, cap: int = 5) -> float:
        """sublinear gain: up to 'cap' units; beyond that, saturate."""
        return min(cap, hits) * unit

    def tag_bonus(self, doc: Document, query_terms: set, unit: float = 0.5) -> float:
        md = doc.metadata or {}
        tags = md.get("tags", []) or []
        if not tags:
            return 0.0
        tag_weights = md.get("tag_weights", {}) or {}
        literal = any(t in query_terms for t in tags)
        if literal and tag_weights:
            w = sum(float(tag_weights.get(t, 0.0)) for t in tags if t in query_terms)
            return min(1.5, w * unit)
        return unit if literal else 0.0

    def calculate_score(self, doc: Document, query_kws: set) -> float:
        """Compute metadata score in [0,1] (higher = better)."""
        md = (doc.metadata or {})

        fl = self.kw_hits(query_kws, md.get("first_line", ""))
        sh = self.kw_hits(query_kws, md.get("sub_header", ""))
        tt = self.kw_hits(query_kws, md.get("toc_title", ""))

        boost = 0.0
        if fl:
            boost += self.soft_gain(fl, self.first_line_boost, cap=3)
        if sh:
            boost += self.soft_gain(sh, self.sub_header_boost, cap=3)
        if tt:
            boost += self.soft_gain(tt, self.toc_title_boost, cap=3)

        kw_hits = self.overlap_hits(query_kws, md.get("keywords", []))
        if kw_hits:
            boost += self.soft_gain(kw_hits, self.keywords_boost, cap=6)

        boost += self.tag_bonus(doc, query_kws, unit=self.tag_unit_bonus)

        boost += float(self.layer_bias.get(md.get("layer"), 0.0))

        boost = min(self.metadata_total_cap, max(0.0, boost))
        return boost / self.metadata_total_cap if self.metadata_total_cap > 0 else 0.0


### cross endcoder score

In [ ]:
class CrossEncoderScore:
    def __init__(self, model_name="cross-encoder/ms-marco-MiniLM-L-6-v2", max_doc_chars=2800):
        self.model_name = model_name
        self.max_doc_chars = max_doc_chars
        self._ce = None

    def load_model(self):
        if self._ce is None:
            try:
                from sentence_transformers import CrossEncoder
                self._ce = CrossEncoder(self.model_name)
            except ImportError:
                raise RuntimeError("sentence_transformers is required. Install with: pip install -U sentence-transformers")
        return self._ce

    def compose_text(self, d: Document) -> str:
        return (d.page_content or "")[:self.max_doc_chars]

    def calculate_score(self, query: str, document: Document) -> float:
        """Calculate cross-encoder score for a single document."""
        ce = self.load_model()
        pair = [query, self.compose_text(document)]
        ce_score = ce.predict([pair])[0]
        return float(ce_score)


In [ ]:
from types import SimpleNamespace
doc = SimpleNamespace(page_content="This document is about network security, phishing, and MFA best practices.")
cs = CrossEncoderScore(
    model_name="Pongsasit/mod-th-cross-encoder-minilm",
    max_doc_chars=2800
)

print(cs.calculate_score("ใน Cyber security อธิบาย phishing คืออะไร", doc))


### fusion score

In [ ]:
# Score Fusion Component
class ScoreFusion:
    def __init__(self, ce_weight=0.55, meta_weight=0.35, vec_weight=0.10, dist_weight=0.15):
        self.ce_weight = ce_weight
        self.meta_weight = meta_weight
        self.vec_weight = vec_weight
        self.dist_weight = dist_weight

    def fuse_scores(
        self,
        query: str,
        results_with_vec_scores: List[Tuple[Document, float, float]],
        metadata_score: MetadataScore,
        cross_encoder_score: CrossEncoderScore,
    ) -> List[Tuple[Document, float]]:
        """
        results_with_vec_scores: [(doc, vec_distance_or_score_lower_is_better, distribution_weight)]
        Returns list of (doc, final_score) sorted desc by final_score.
        """
        docs = [d for d, _, _ in results_with_vec_scores]
        vec_raw = [s for _, s, _ in results_with_vec_scores]
        dist_weights = [w for _, _, w in results_with_vec_scores]

        # Calculate cross-encoder scores for each document
        ce_scores_raw = [cross_encoder_score.calculate_score(query, doc) for doc in docs]
        ce_norm = TextProcessingUtils.minmax_normalize(ce_scores_raw)

        vec_sim = [1.0 / (1.0 + max(0.0, v)) for v in vec_raw]
        vec_norm = TextProcessingUtils.minmax_normalize(vec_sim)

        dist_norm = TextProcessingUtils.minmax_normalize(dist_weights)

        kws = set(TextProcessingUtils.extract_keywords(query))
        meta_norm = [metadata_score.calculate_score(d, kws) for d in docs]

        w_sum = max(1e-9, self.ce_weight + self.meta_weight + self.vec_weight + self.dist_weight)
        fused = []
        for d, ce, me, ve, di in zip(docs, ce_norm, meta_norm, vec_norm, dist_norm):
            final = (self.ce_weight * ce +
                     self.meta_weight * me +
                     self.vec_weight * ve +
                     self.dist_weight * di) / w_sum
            fused.append((d, float(final)))

        fused.sort(key=lambda x: x[1], reverse=True)
        return fused



## cross endcoder retriever

### cross endcoder rerank

In [ ]:
from typing import Any, Dict, List, Optional, Tuple
from langchain.schema import Document
from sentence_transformers import CrossEncoder

class CrossEncoderRetriever(BaseRetriever):
    """
    Retriever that performs a distributed vector search and then reranks the results
    using only the score from a cross-encoder model.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 rag_system: MultiDocumentRAG,
                 distributed_searcher: DistributedSearcher,
                 query_rewriter=None,
                 ce_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
                 max_doc_chars: int = 2800,
                 **kwargs):
        super().__init__(**kwargs)
        self.rag_system = rag_system
        self.K = kwargs.get('K', 20)
        self.k = kwargs.get('k', 4)
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)
        self.query_rewriter = query_rewriter

        # Injected components
        self.distributed_searcher = distributed_searcher

        # Cross-encoder attributes
        self.ce_model_name = ce_model_name
        self.max_doc_chars = max_doc_chars
        self._ce = None

    def _load_ce(self) -> CrossEncoder:
        """Lazily load the cross-encoder model."""
        if self._ce is None:
            print(f"Loading CrossEncoder model: {self.ce_model_name}")
            self._ce = CrossEncoder(self.ce_model_name)
        return self._ce

    def _compose_text(self, d: Document) -> str:
        """Compose text for cross-encoder from a Document."""
        return (d.page_content or "")[:self.max_doc_chars]

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """
        Main entry point for retrieving and reranking documents.
        """
        search_query = query
        distributions = None

        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_query, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Distributions: {distributions}")
            print(f"🔄 Rewritten Query: {search_query}")

        return self._get_relevant_documents_with_preprocessed_data(
            original_query=query,
            search_query=search_query,
            distributions=distributions
        )

    def _get_relevant_documents_with_preprocessed_data(
        self,
        original_query: str,
        search_query: str,
        distributions: Optional[List[Dict[str, Any]]],
        K: Optional[int] = None,
        k: Optional[int] = None
    ) -> List[Document]:
        """
        Retrieves and reranks documents based on pre-processed query data.
        """
        retrieve_K = K if K is not None else self.K
        return_k = k if k is not None else self.k

        # Step 1: Distributed search to get candidate documents
        results_with_scores = self.distributed_searcher.search_with_scores(
            search_query, retrieve_K, distributions
        )
        if not results_with_scores or not results_with_scores[0]:
            print("No documents found in distributed search")
            return []


        candidate_docs = [doc for doc, score, weight in results_with_scores]
        print(f"\\n🔍 Found {len(candidate_docs)} candidate documents for reranking.")

        # Step 2: Cross-encoder reranking
        if not candidate_docs:
            return []

        ce = self._load_ce()
        pairs = [[original_query, self._compose_text(d)] for d in candidate_docs]
        scores = ce.predict(pairs)

        reranked = sorted(zip(candidate_docs, scores), key=lambda x: float(x[1]), reverse=True)

        print(f"\\n🧠 Cross-encoder reranking completed. Returning top {return_k} documents:")
        for i, (doc, score) in enumerate(reranked[:return_k], 1):
            m = doc.metadata or {}
            print(f"#{i}: [{m.get('course_name')}] {m.get('toc_title')} (CE score: {score:.4f})")

        return [d for d, _ in reranked[:return_k]]

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)


### multi cross endcoder rerank

In [ ]:
class MultiCrossEncoderRetriever(BaseRetriever):
    """
    A retriever that wraps CrossEncoderRetriever to handle multiple atomic queries.
    It calls the wrapped retriever for each query and combines the unique results without further reranking.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 retriever: CrossEncoderRetriever,
                 rag_system: MultiDocumentRAG,
                 query_rewriter=None, # Expects a MultiQueryRewriter instance
                 max_queries: int = 5,
                 **kwargs):
        super().__init__(**kwargs)
        self.retriever = retriever
        self.rag_system = rag_system
        self.query_rewriter = query_rewriter
        self.K = kwargs.get('K', 20)  # Initial retrieval count for each atomic query
        self.k = kwargs.get('k', 4)   # Docs to return per atomic query before combining
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)
        self.max_queries = max_queries

    def _get_relevant_documents(self, query: str) -> List[Document]:
        search_queries = [query]
        distributions = None

        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_queries, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Atomic queries: {search_queries}")
            print(f"Distributions: {distributions}")

            if len(search_queries) > self.max_queries:
                print(f"Limiting queries from {len(search_queries)} to {self.max_queries}")
                search_queries = search_queries[:self.max_queries]

        if len(search_queries) == 1:
            # If only one query, just use the injected retriever directly
            return self.retriever._get_relevant_documents_with_preprocessed_data(
                original_query=query,
                search_query=search_queries[0],
                distributions=distributions,
                K=self.K,
                k=self.k
            )

        # Process multiple queries and collect unique results
        all_results = []
        doc_ids = set()
        k_per_query = self.k

        for i, search_query in enumerate(search_queries):
            print(f"\\n--- Processing Atomic Query {i+1}/{len(search_queries)}: '{search_query}' ---")
            query_docs = self.retriever._get_relevant_documents_with_preprocessed_data(
                original_query=query, # Use original query for CE scoring consistency
                search_query=search_query,
                distributions=distributions,
                K=self.K,
                k=k_per_query
            )

            for doc in query_docs:
                doc_id = str(hash(doc.page_content))
                if doc_id not in doc_ids:
                    doc_ids.add(doc_id)
                    all_results.append(doc)

            print(f"Retrieved {len(query_docs)} documents for this query.")

        print(f"\\nReturning {len(all_results)} unique documents from {len(search_queries)} queries.")
        for i, doc in enumerate(all_results, 1):
            m = doc.metadata or {}
            print(f"Final Doc #{i}: [{m.get('course_name')}] {m.get('toc_title')}")

        return all_results

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)

## cross encoder with metadata

### cross endcoder with metadata

In [ ]:
class MetadataCrossEncoderRetriever(BaseRetriever):
    """
    Retriever with cross-encoder reranking + metadata-aware fusion.

    This class now accepts injected components for each logical part of the retrieval process.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 rag_system: MultiDocumentRAG,
                 distributed_searcher: DistributedSearcher,
                 metadata_score: MetadataScore,
                 cross_encoder_score: CrossEncoderScore,
                 score_fusion: ScoreFusion,
                 query_rewriter=None,
                 **kwargs):
        super().__init__(**kwargs)
        self.rag_system = rag_system
        self.K = kwargs.get('K', 20)
        self.k = kwargs.get('k', 4)
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)
        self.query_rewriter = query_rewriter

        # Injected components
        self.distributed_searcher = distributed_searcher
        self.metadata_score = metadata_score
        self.cross_encoder_score = cross_encoder_score
        self.score_fusion = score_fusion

    def _get_relevant_documents(self, query: str) -> List[Document]:
        search_query = query
        distributions = None
        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_query, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Distributions: {distributions}")
            print(f"🔄 Rewritten Query: {search_query}")

        # Step 1: Distributed search
        results_with_scores = self.distributed_searcher.search_with_scores(
            search_query, self.K, distributions
        )
        if not results_with_scores:
            print("No documents found in distributed search")
            return []

        # Step 2: Score fusion (which internally calls cross-encoder and metadata scoring)
        fused = self.score_fusion.fuse_scores(
            query,
            results_with_scores,
            self.metadata_score,
            self.cross_encoder_score
        )

        print(f"\n Returning top {self.k} by fused score (w_ce={self.score_fusion.ce_weight}, w_meta={self.score_fusion.meta_weight}, w_vec={self.score_fusion.vec_weight}, w_dist={self.score_fusion.dist_weight})")
        for i, (doc, s) in enumerate(fused[:self.k], 1):
            m = doc.metadata or {}
            print(f"#{i}: [{m.get('course_name')}] {m.get('toc_title')}  (fused={s:.4f})")

        return [d for d, _ in fused[:self.k]]

    def _get_relevant_documents_with_preprocessed_data(
        self,
        original_query: str,
        search_query: str,
        distributions: Optional[List[Dict[str, Any]]],
        K: Optional[int] = None,
        k: Optional[int] = None
    ) -> List[Document]:

        # Use provided K and k values or fall back to instance properties
        retrieve_K = K if K is not None else self.K
        return_k = k if k is not None else self.k

        # Step 1: Distributed search
        results_with_scores = self.distributed_searcher.search_with_scores(
            search_query, retrieve_K, distributions
        )
        if not results_with_scores:
            print("No documents found in distributed search")
            return []

        # Step 2: Score fusion (which internally calls cross-encoder and metadata scoring)
        fused = self.score_fusion.fuse_scores(
            original_query,  # Use original query for cross-encoder scoring
            results_with_scores,
            self.metadata_score,
            self.cross_encoder_score
        )

        print(f"\nReturning top {return_k} by fused score (w_ce={self.score_fusion.ce_weight}, w_meta={self.score_fusion.meta_weight}, w_vec={self.score_fusion.vec_weight}, w_dist={self.score_fusion.dist_weight})")
        for i, (doc, s) in enumerate(fused[:return_k], 1):
            m = doc.metadata or {}
            print(f"#{i}: [{m.get('course_name')}] {m.get('toc_title')}  (fused={s:.4f})")

        return [d for d, _ in fused[:return_k]]

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)


### multi cross encoder with metadata

In [ ]:

class MultiMetadataCrossEncoderRetriever(BaseRetriever):
    """
    Retriever that wraps a MetadataCrossEncoderRetriever to handle multiple queries efficiently.
    This class processes multiple queries by calling the wrapped retriever for each query
    and combining the results without additional reranking.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 retriever: MetadataCrossEncoderRetriever,  # Inject the retriever
                 rag_system: MultiDocumentRAG,  # Also need rag_system for query rewriting
                 query_rewriter=None,
                 max_queries=5,  # Limit the number of atomic queries to process
                 **kwargs):
        super().__init__(**kwargs)
        self.retriever = retriever
        self.rag_system = rag_system
        self.K = kwargs.get('K', 20)  # Total docs to retrieve before reranking
        self.k = kwargs.get('k', 4)   # Final number of docs to return per query
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)
        self.query_rewriter = query_rewriter
        self.max_queries = max_queries

    def _get_relevant_documents(self, query: str) -> List[Document]:
        search_queries = [query]  # Default to single query
        distributions = None

        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_queries, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Atomic queries: {search_queries}")
            print(f"Distributions: {distributions}")

            # Limit the number of queries to process
            if len(search_queries) > self.max_queries:
                print(f"Limiting queries from {len(search_queries)} to {self.max_queries}")
                search_queries = search_queries[:self.max_queries]

        # If only one query, just use the injected retriever directly
        if len(search_queries) == 1:
            # Check if the retriever has the preprocessed method
            if hasattr(self.retriever, '_get_relevant_documents_with_preprocessed_data'):
                return self.retriever._get_relevant_documents_with_preprocessed_data(
                    query, search_queries[0], distributions, self.K, self.k
                )
            else:
                # Fallback to the standard method
                return self.retriever._get_relevant_documents(search_queries[0])

        # For multiple queries, call the retriever for each query and collect all results
        all_results = []
        doc_ids = set()

        # Calculate how many documents to retrieve per query
        k_per_query = self.k  # Use k directly for each query

        # Process each query independently
        for i, search_query in enumerate(search_queries):
            # Get documents for this query
            if hasattr(self.retriever, '_get_relevant_documents_with_preprocessed_data'):
                query_docs = self.retriever._get_relevant_documents_with_preprocessed_data(
                    query, search_query, distributions, self.K, k_per_query
                )
            else:
                # Fallback to the standard method
                # Temporarily modify the retriever's k value
                original_k = self.retriever.k
                self.retriever.k = k_per_query
                query_docs = self.retriever._get_relevant_documents(search_query)
                self.retriever.k = original_k  # Restore original value

            # Add to results, avoiding duplicates
            for doc in query_docs:
                doc_id = doc.metadata.get("doc_id", str(hash(doc.page_content)))
                if doc_id not in doc_ids:
                    doc_ids.add(doc_id)
                    all_results.append(doc)

            print(f"Retrieved {len(query_docs)} documents for query {i+1}/{len(search_queries)}")

        # No reranking - just return all the collected documents
        result = all_results

        print(f"\n Returning {len(result)} documents from {len(search_queries)} queries (k={k_per_query} per query)")
        for i, doc in enumerate(result, 1):
            m = doc.metadata or {}
            print(f"#{i}: [{m.get('course_name')}] {m.get('toc_title')}")

        return result

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)

## metadata retrival

### metadata retrival

In [ ]:
from typing import Any, Dict, List, Optional, Tuple
from langchain.schema import Document
import numpy as np

# Utility class, assuming TextProcessingUtils is defined elsewhere as in the notebook
# from your_utils_module import TextProcessingUtils

class MetadataRetriever(BaseRetriever):
    """
    Retriever that combines distributed vector search with metadata-based reranking.
    The final score is a weighted combination of the initial vector similarity and a calculated metadata score.
    It does not use a cross-encoder or complex score fusion.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 rag_system: MultiDocumentRAG,
                 distributed_searcher: DistributedSearcher,
                 metadata_score: MetadataScore,
                 query_rewriter=None,
                 meta_weight: float = 0.6,
                 vec_weight: float = 0.4,
                 **kwargs):
        super().__init__(**kwargs)
        self.rag_system = rag_system
        self.K = kwargs.get('K', 20)
        self.k = kwargs.get('k', 4)
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)
        self.query_rewriter = query_rewriter

        # Injected components
        self.distributed_searcher = distributed_searcher
        self.metadata_score = metadata_score

        # Reranking weights
        self.meta_weight = meta_weight
        self.vec_weight = vec_weight

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """
        Main entry point for retrieving documents, including query rewriting.
        """
        search_query = query
        distributions = None

        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_query, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Distributions: {distributions}")
            print(f"Rewritten Query: {search_query}")

        return self._get_relevant_documents_with_preprocessed_data(
            original_query=query,
            search_query=search_query,
            distributions=distributions
        )

    def _get_relevant_documents_with_preprocessed_data(
        self,
        original_query: str,
        search_query: str,
        distributions: Optional[List[Dict[str, Any]]],
        K: Optional[int] = None,
        k: Optional[int] = None
    ) -> List[Document]:
        """
        Retrieves and reranks documents based on pre-processed query data.
        """
        retrieve_K = K if K is not None else self.K
        return_k = k if k is not None else self.k

        # Step 1: Distributed search
        results_with_scores = self.distributed_searcher.search_with_scores(
            search_query, retrieve_K, distributions
        )
        if not results_with_scores:
            print("No documents found in distributed search")
            return []

        # Step 2: Rerank using only vector and metadata scores
        docs = [d for d, _, _ in results_with_scores]
        vec_raw_distances = [s for _, s, _ in results_with_scores]

        # Normalize vector distances to similarity scores [0, 1]
        vec_sim = [1.0 / (1.0 + max(0.0, v)) for v in vec_raw_distances]
        vec_norm = TextProcessingUtils.minmax_normalize(vec_sim)

        # Calculate metadata scores [0, 1]
        query_kws = set(TextProcessingUtils.extract_keywords(original_query))
        meta_scores = [self.metadata_score.calculate_score(d, query_kws) for d in docs]

        # Fuse the two scores
        final_scores = []
        w_sum = max(1e-9, self.vec_weight + self.meta_weight)
        for i in range(len(docs)):
            score = (self.vec_weight * vec_norm[i] + self.meta_weight * meta_scores[i]) / w_sum
            final_scores.append((docs[i], score))

        # Sort by the final fused score
        final_scores.sort(key=lambda x: x[1], reverse=True)

        print(f"\\nReturning top {return_k} by metadata score (w_vec={self.vec_weight}, w_meta={self.meta_weight})")
        for i, (doc, s) in enumerate(final_scores[:return_k], 1):
            m = doc.metadata or {}
            print(f"#{i}: [{m.get('course_name')}] {m.get('toc_title')}  (fused_meta_score={s:.4f})")

        return [d for d, _ in final_scores[:return_k]]

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)

### multi metadata revtrival

In [ ]:
class MultiMetadataRetriever(BaseRetriever):
    """
    A retriever that wraps MetadataRetriever to handle multiple atomic queries efficiently.
    It calls the wrapped retriever for each query and combines the unique results.
    """

    class Config:
        arbitrary_types_allowed = True
        extra = "allow"

    def __init__(self,
                 retriever: MetadataRetriever,
                 rag_system: MultiDocumentRAG,
                 query_rewriter=None, # Expects a MultiQueryRewriter instance
                 max_queries: int = 5,
                 **kwargs):
        super().__init__(**kwargs)
        self.retriever = retriever
        self.rag_system = rag_system
        self.query_rewriter = query_rewriter
        self.K = kwargs.get('K', 20)
        self.k = kwargs.get('k', 4)
        self.use_query_rewrite = kwargs.get('use_query_rewrite', True)
        self.max_queries = max_queries

    def _get_relevant_documents(self, query: str) -> List[Document]:
        search_queries = [query]
        distributions = None

        if self.use_query_rewrite and self.query_rewriter:
            headers_text = self.query_rewriter.get_default_headers_text()
            course_list = list(self.rag_system.courses.keys())
            search_queries, distributions = self.query_rewriter.rewrite_and_distribute(
                question=query,
                headers_text=headers_text,
                course_list=course_list,
            )
            print(f"Atomic queries: {search_queries}")
            print(f"Distributions: {distributions}")

            if len(search_queries) > self.max_queries:
                print(f"Limiting queries from {len(search_queries)} to {self.max_queries}")
                search_queries = search_queries[:self.max_queries]

        if len(search_queries) == 1:
            return self.retriever._get_relevant_documents_with_preprocessed_data(
                original_query=query,
                search_query=search_queries[0],
                distributions=distributions,
                K=self.K,
                k=self.k
            )

        all_results = []
        doc_ids = set()
        k_per_query = self.k

        for i, search_query in enumerate(search_queries):
            query_docs = self.retriever._get_relevant_documents_with_preprocessed_data(
                original_query=query,
                search_query=search_query,
                distributions=distributions,
                K=self.K,
                k=k_per_query
            )

            for doc in query_docs:
                doc_id = str(hash(doc.page_content))
                if doc_id not in doc_ids:
                    doc_ids.add(doc_id)
                    all_results.append(doc)

            print(f"Retrieved {len(query_docs)} documents for query {i+1}/{len(search_queries)}")

        print(f"\\nReturning {len(all_results)} unique documents from {len(search_queries)} queries (k={k_per_query} per query)")
        for i, doc in enumerate(all_results, 1):
            m = doc.metadata or {}
            print(f"#{i}: [{m.get('course_name')}] {m.get('toc_title')}")

        return all_results

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)


## embedding and store

### metadata

In [ ]:
def setup_example_courses():

    default_section_titles = [
    '1. รหัสและชื่อหลักสูตร',
    '9. ชื่อ เลขประจ าตัวบัตรประชาชน ต าแหน่ง และคุณวุฒิการศึกษาของอาจารย์ผู้รับผิดชอบหลักสูตร',
    '10. สถานที่จัดการเรียนการสอน',
    '11. สถานการณ์ภายนอกหรือการพัฒนาที่จ าเป็นต้องน ามาพิจารณาในการวางแผนหลักสูตร',
    '12. ผลกระทบจากข้อ 11 ต่อการพัฒนาหลักสูตรและความเกี่ยวข้องกับพันธกิจของสถาบัน',
    '13. ความสัมพันธ์ (ถ้ามี) กับหลักสูตรอื่น ที่เปิดสอนในคณะ/สาขาวิชาอื่นของสถาบัน',

    '1. ปรัชญา ความส าคัญ และวัตถุประสงค์ของหลักสูตร',
    '2. แผนพัฒนาปรับปรุง',

    '1. ระบบการจัดการศึกษา',
    '2. การด าเนินการหลักสูตร',
    '3. หลักสูตรและอาจารย์ผู้สอน',
    '4. องค์ประกอบเกี่ยวกับประสบการณ์ภาคสนาม (การฝึกงานหรือสหกิจศึกษา)',
    '5. ข้อก าหนดเกี่ยวกับการท าโครงงานหรืองานวิจัย (ถ้ามี)',

    '1. การพัฒนาคุณลักษณะพิเศษของนักศึกษา',
    '2. การพัฒนาผลลัพธ์การเรียนรู้ของหลักสูตร (PLOs)',
    '3. ตารางความสัมพันธ์ระหว่างรายวิชาและผลลัพธ์การเรียนรู้ของหลักสูตร (TQF 5 ด้าน)',

    '1. กฎระเบียบหรือหลักเกณฑ์ในการให้ระดับคะแนน',
    '2. กระบวนการทวนสอบมาตรฐานผลสัมฤทธิ์ของนักศึกษา',
    'เกณฑ์การส าเร็จการศึกษาตามหลักสูตร',

    '1. การเตรียมการส าหรับอาจารย์ใหม่',
    '2. การพัฒนาความรู้และทักษะให้แก่อาจารย์',

    '1. การบริหารหลักสูตร',
    '2. บัณฑิต',
    '3. นักศึกษา',
    '4. อาจารย์',
    '5. หลักสูตร การเรียนการสอน การประเมินผู้เรียน',
    '6. สิ่งสนับสนุนการเรียนรู้',
    '7. การบริหารบุคลากรสนับสนุนการเรียนการสอน',
    '8. การสนับสนุนและการให้ค าแนะน านักศึกษา',
    '9. ความต้องการของตลาดแรงงาน สังคม และ/หรือความพึงพอใจของผู้ใช้บัณฑิต',
    '10. ตัวบ่งชี้ผลการด าเนินงาน (Key Performance Indicators)',

    '1. การประเมินประสิทธิผลของการสอน',
    '2. การประเมินหลักสูตรในภาพรวม',
    '3. การประเมินผลการด าเนินงานตามรายละเอียดหลักสูตร',
    '4. การทบทวนผลการประเมินและวางแผนปรับปรุง',

    'ภาคผนวก ก', 'ภาคผนวก ข', 'ภาคผนวก ค', 'ภาคผนวก ง', 'ภาคผนวก จ',
    'ภาคผนวก ฉ', 'ภาคผนวก ซ', 'ภาคผนวก ฌ', 'ภาคผนวก ญ',
    ]

    # Default selected headers
    default_selected = [
        '1. รหัสและชื่อหลักสูตร',
        '3. หลักสูตรและอาจารย์ผู้สอน',
        'เกณฑ์การส าเร็จการศึกษาตามหลักสูตร',
        '5. ข้อก าหนดเกี่ยวกับการท าโครงงานหรืองานวิจัย (ถ้ามี)',
        # 'ภาคผนวก ข',
    ]

    # Default chunking plan
    default_chunking_plan = {
        "3. หลักสูตรและอาจารย์ผู้สอน": {"method": "subtoc", "metadata": {"program": "CS"}},

        # "ภาคผนวก ข": {
        #     "method": "regex",
        #     "data": {
        #         "pattern": r'(\d+)\.\s*((?:นาย|นาง|นางสาว)[^\n]+)\s*\n\s*((?:Mr\.|Ms\.|Miss)[^\n]+)',
        #         "remove_between": [{
        #             "start": r"ผลงานทางวิชาการ",
        #             "end": r"ประสบการณ์การสอน",
        #             "inclusive": True,
        #             "only_first": True
        #         }]
        #     }
        # },

        "__default__": {"method": "semantic", "metadata": {"fallback": True}},
    }

    # Example subheader map
    default_subheader_map = {
        "3. หลักสูตรและอาจารย์ผู้สอน":[
        "1. ระบบการจัดการศึกษา",
        "3. หลักสูตรและอาจารย์ผู้สอน",
        "3.1.3.2 หมวดวิชาเฉพาะ (Core Courses)",
        "(Required Courses)",
        "ค าอธิบายระบบรหัสวิชา",
        "ปีที่ 1 ภาคการศึกษาที่ 1",
        "ปีที่ 1 ภาคการศึกษาที่ 2",
        "ปีที่ 2 ภาคการศึกษาที่ 1",
        "ปีที่ 2 ภาคการศึกษาที่ 2",
        "ปีที่ 3 ภาคการศึกษาที่ 1",
        "ปีที่ 3 ภาคการศึกษาที่ 2",
        "ปีที่ 4 ภาคการศึกษาที่ 1",
        "ปีที่ 4 ภาคการศึกษาที่ 2",
        "ค าอธิบายรายวิชา",
        "3.2 ชื่อ เลขประจ าตัวบัตรประชาชน ต าแหน่งและคุณวุฒิของอาจารย์",
        "4. องค์ประกอบเกี่ยวกับประสบการณ์ภาคสนาม (การฝึกงานหรือสหกิจศึกษา)",
        "4.2 ช่วงเวลา",
        "4.3 การจัดเวลาและตารางสอน",
        "5. ข้อก าหนดเกี่ยวกับการท าโครงงานหรืองานวิจัย (ถ้ามี)",
        "5.3 ช่วงเวลา",
        "5.4 จ านวนหน่วยกิต"
        ]
        ,
    }

    return {
        'section_titles': default_section_titles,
        'selected_headers': default_selected,
        'chunking_plan': default_chunking_plan,
        'subheader_map': default_subheader_map
    }

In [ ]:
def setup_ai_courses():

    default_section_titles = [

        'โครงสร้างหลักสูตร รายวิชาหรือชุดวิชา และจ านวนหน่วยกิต',
        'โครงสร้างหลักสูตร',
        'การด าเนินการหลักสูตร',
        'ตัวอย่างแผนการศึกษา',
        'ค าอธิบายรายวิชา',
        '4. การจัดกระบวนการเรียนรู้',
        '5. การประเมินผลการเรียนและเกณฑ์การส าเร็จการศึกษา',
        '5.1 การประเมินผลการเรียน',
        '5.2 เกณฑ์การส าเร็จการศึกษา',
    ]

    # Default selected headers
    default_selected = [
        'โครงสร้างหลักสูตร รายวิชาหรือชุดวิชา และจ านวนหน่วยกิต',
        'โครงสร้างหลักสูตร',
        'การด าเนินการหลักสูตร',
        'ตัวอย่างแผนการศึกษา',
        'ค าอธิบายรายวิชา',
        '5.2 เกณฑ์การส าเร็จการศึกษา',
    ]

    # Default chunking plan
    default_chunking_plan = {
        "โครงสร้างหลักสูตร": {"method": "subtoc", "metadata": {"program": "CS"}},
        "ตัวอย่างแผนการศึกษา": {"method": "subtoc", "metadata": {"program": "CS"}},

        # "ภาคผนวก ข": {
        #     "method": "regex",
        #     "data": {
        #         "pattern": r'(\d+)\.\s*((?:นาย|นาง|นางสาว)[^\n]+)\s*\n\s*((?:Mr\.|Ms\.|Miss)[^\n]+)',
        #         "remove_between": [{
        #             "start": r"ผลงานทางวิชาการ",
        #             "end": r"ประสบการณ์การสอน",
        #             "inclusive": True,
        #             "only_first": True
        #         }]
        #     }
        # },

        "__default__": {"method": "semantic", "metadata": {"fallback": True}},
    }

    # Example subheader map
    default_subheader_map = {
        "โครงสร้างหลักสูตร":[
        "หมวดวิชาศึกษาทั่วไป (General Education Courses)",
        "กลุ่มวิชามนุษยศาสตร์และสังคมศาสตร์",
        "หมวดวิชาเฉพาะ (Core Courses)",
        "วิชาเฉพาะด้าน (Required Courses)",
        "กลุ่มวิชาโครงงาน หรือสหกิจศึกษา",
        "วิชาเลือกเฉพาะทาง",
        "กลุ่มวิชาเลือกสาขาวิชา",
        "หมวดวิชาเลือกสาขา",
        "ค าอธิบายระบบรหัสวิชา",
        ],
        "ตัวอย่างแผนการศึกษา":[
        "ปีที่ 1 ภาคการศึกษาที่ 1",
        "ปีที่ 1 ภาคการศึกษาที่ 2",
        "ปีที่ 2 ภาคการศึกษาที่ 1",
        "ปีที่ 2 ภาคการศึกษาที่ 2",
        "ปีที่ 3 ภาคการศึกษาที่ 1",
        "ปีที่ 3 ภาคการศึกษาที่ 2",
        "ปีที่ 4 ภาคการศึกษาที่ 1",
        "ปีที่ 4 ภาคการศึกษาที่ 2",
        ]
        ,
    }

    return {
        'section_titles': default_section_titles,
        'selected_headers': default_selected,
        'chunking_plan': default_chunking_plan,
        'subheader_map': default_subheader_map
    }

### buliding

In [ ]:
cs_config = BaselineCourseConfig(
    name="Computer Science",
    file_path="/content/cs_course.txt",
    elective_file_path="content/el.txt",
    offset=9800
)

cy_config = BaselineCourseConfig(
    name="Cyber security",
    file_path="/content/cb_course.txt",
    elective_file_path="content/el.txt",
    offset=2000
)


ai_config = BaselineCourseConfig(
    name="Artificial Intelligence",
    file_path="/content/ai_course.txt",
    elective_file_path="content/el.txt",
    offset=2000
)

it_config = BaselineCourseConfig(
    name="Information Technology",
    file_path="/content/it_course.txt",
    elective_file_path="content/el.txt",
    offset=2000
)

gis_config = BaselineCourseConfig(
    name="Geo-Informatics",
    file_path="/content/gis_course.txt",
    elective_file_path="content/el.txt",
    offset=2000
)


baseline_rag = BaselineRAG(enable_metadata=True)

baseline_rag.add_course(cs_config)
baseline_rag.add_course(cy_config)
baseline_rag.add_course(ai_config)
baseline_rag.add_course(it_config)
baseline_rag.add_course(gis_config)

baseline_rag.build_vectorstores()
baseline_rag.save_vectorstores("./baseline_vectordb")

import shutil
import os

# Define the directory containing the vector database
vector_db_dir = "/content/baseline_vectordb"  # Replace with the actual path to your vectordb directory
zip_filename = "baseline_vectordb.zip"
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', vector_db_dir)

print(f"Vector database successfully zipped to: {zip_filename}")

In [ ]:
defaults = setup_example_courses()
ai = setup_ai_courses()

with open('/content/el.txt', 'r', encoding='utf-8') as f:
    electives_text = f.read()
catalog = ElectiveCatalog.from_text(electives_text)

cs_course = CourseConfig(
    name="Computer Science",
    file_path="/content/cs_course.txt",
    offset=9800,
    section_titles=defaults['section_titles'],
    selected_headers=defaults['selected_headers'],
    subheader_map=defaults['subheader_map'],
    chunking_plan=defaults['chunking_plan']
)
cs_course.program_name = "วิทยาการคอมพิวเตอร์"
cs_course.electives_catalog = catalog
cs_course.electives_label = "วิชาเลือกเพิ่มเติมที่สามารถลงได้"

cy_course = CourseConfig(
    name="Cyber security",
    file_path="/content/cy-corurse.txt",
    offset=9800,
    section_titles=defaults['section_titles'],
    selected_headers=defaults['selected_headers'],
    subheader_map=defaults['subheader_map'],
    chunking_plan=defaults['chunking_plan']
)
cy_course.program_name = "ความมั่นคงปลอดภัยไซเบอร์"
cy_course.electives_catalog = catalog
cy_course.electives_label = "วิชาเลือกเพิ่มเติมที่สามารถลงได้"

ai_course = CourseConfig(
    name="Artificial Intelligence",
    file_path="/content/ai_course.txt",
    offset=2000,
    section_titles=ai['section_titles'],
    selected_headers=ai['selected_headers'],
    subheader_map=ai['subheader_map'],
    chunking_plan=ai['chunking_plan']
)
ai_course.program_name = "ปัญญาประดิษฐ์"
ai_course.electives_catalog = catalog
ai_course.electives_label = "วิชาเลือกเพิ่มเติมที่สามารถลงได้"


gis_course = CourseConfig(
    name="Geo-Informatics",
    file_path="/content/gis_course.txt",
    offset=2000,
    section_titles=defaults['section_titles'],
    selected_headers=defaults['selected_headers'],
    subheader_map=defaults['subheader_map'],
    chunking_plan=defaults['chunking_plan']
)
cy_course.program_name = "ภูมิสารสนเทศศาสตร์"
cy_course.electives_catalog = catalog
cy_course.electives_label = "วิชาเลือกเพิ่มเติมที่สามารถลงได้"

it_course = CourseConfig(
    name="Information Technology",
    file_path="/content/it_course.txt",
    offset=2000,
    section_titles=defaults['section_titles'],
    selected_headers=defaults['selected_headers'],
    subheader_map=defaults['subheader_map'],
    chunking_plan=defaults['chunking_plan']
)
cy_course.program_name = "เทคโนโลยีสารสนเทศ"
cy_course.electives_catalog = catalog
cy_course.electives_label = "วิชาเลือกเพิ่มเติมที่สามารถลงได้"

rag_build = MultiDocumentRAG()


rag_build.add_course(cy_course)
rag_build.add_course(cs_course)
rag_build.add_course(ai_course)
rag_build.add_course(it_course)
rag_build.add_course(gis_course)

# Build vectorstores
rag_build.build_vectorstores()
rag_build.save_vectorstores("./rag_vectordb")

vector_db_dir = "/content/rag_vectordb"
zip_filename = "rag_vectordb.zip"
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', vector_db_dir)

print(f"Vector database successfully zipped to: {zip_filename}")

### load save

In [ ]:
import shutil
import os

def unzip_vectordb(zip_filepath: str, extract_dir: str):
    """
    Unzips a vector database zip file to a specified directory.

    Args:
        zip_filepath: The path to the vector database zip file.
        extract_dir: The directory to extract the contents to.
    """
    try:
        shutil.unpack_archive(zip_filepath, extract_dir, 'zip')
        print(f"Successfully unzipped {zip_filepath} to {extract_dir}")
    except FileNotFoundError:
        print(f"Error: Zip file not found at {zip_filepath}")
    except shutil.ReadError:
        print(f"Error: Could not read the zip file {zip_filepath}. It might be corrupted or not a valid zip file.")
    except Exception as e:
        print(f"An error occurred during unzipping: {e}")

# Example usage (you can modify this as needed)
zip_file = "baseline_vectordb.zip"
destination_folder = "/content/baseline_vectordb"
unzip_vectordb(zip_file, destination_folder)

zip_file = "rag_vectordb.zip"
destination_folder = "/content/rag_vectordb"
unzip_vectordb(zip_file, destination_folder)

In [ ]:
baseline_rag = BaselineRAG()
baseline_rag.load_vectorstores("./baseline_vectordb")

rag = MultiDocumentRAG()
rag.load_vectorstores("./rag_vectordb")

In [ ]:
print("=== Vectorstore sanity ===")
for cname, vs in rag.vectorstores.items():
    ntotal = getattr(vs.index, "ntotal", None)
    ds = getattr(vs, "docstore", None)
    dict_len = len(getattr(ds, "_dict", {})) if getattr(ds, "_dict", None) else "n/a"
    print(f"{cname}: ntotal={ntotal}, docstore_items={dict_len}")
print("=== Vectorstore sanity ===")
for cname, vs in baseline_rag.vectorstores.items():
    ntotal = getattr(vs.index, "ntotal", None)
    ds = getattr(vs, "docstore", None)
    dict_len = len(getattr(ds, "_dict", {})) if getattr(ds, "_dict", None) else "n/a"
    print(f"{cname}: ntotal={ntotal}, docstore_items={dict_len}")

In [ ]:
import os, faiss
for d in os.listdir("baseline_vectordb"):
    p = os.path.join("baseline_vectordb", d, "index.faiss")
    if os.path.isfile(p):
        idx = faiss.read_index(p)
        print(d, "ntotal=", idx.ntotal)


### build components

In [ ]:


baseline_query = BaselineQuery()

query_rewriter = QueryRewriter(
    model="qwen/qwen3-next-80b-a3b-instruct",
    temperature=0.0,
    openai_api_key=OPEN_ROUTER_KEY,
    base_url="https://openrouter.ai/api/v1"
)

multi_query_rewriter = MultiQueryRewriter(
    model="qwen/qwen3-next-80b-a3b-instruct",
    temperature=0.0,
    openai_api_key=OPEN_ROUTER_KEY,
    base_url="https://openrouter.ai/api/v1",
    max_atomic_queries=3
)

distributed_searcher = DistributedSearcher(rag, K=20)
base_distributed_searcher = DistributedSearcher(baseline_rag, K=12)

metadata_score = MetadataScore(
    first_line_boost=0.2,
    sub_header_boost=0.08,
    toc_title_boost=0.1,
    keywords_boost=0.4,
    tag_unit_bonus=0.2,
    layer_bias={"semantic": 0.04, "second": 0.02},
    metadata_total_cap=0.70
)


cross_encoder_score = CrossEncoderScore(
    model_name="Pongsasit/mod-th-cross-encoder-minilm",
    max_doc_chars=2800
)

score_fusion = ScoreFusion(
    ce_weight=0.50,
    meta_weight=0.50,
    vec_weight=0.4,
    dist_weight=0.05
)

### combine base test

In [ ]:
baseline_baseline_baseline = BaselineRetriever(
    rag_system=baseline_rag,
    query_rewriter=baseline_query,
    distributed_searcher=base_distributed_searcher,
    K=12,
    k=4,
    use_query_rewrite=True
)

toc_baseline_baseline       = BaselineRetriever(
    rag_system=rag,
    query_rewriter=baseline_query,
    distributed_searcher=distributed_searcher,
    K=12,
    k=4,
    use_query_rewrite=True
)

baseline_rewriter_baseline= BaselineRetriever(
    rag_system=baseline_rag,
    query_rewriter=query_rewriter,
    distributed_searcher=base_distributed_searcher,
    K=12,
    k=4,
    use_query_rewrite=True
)

baseline_multi_helper = BaselineRetriever(
    rag_system=baseline_rag,
    query_rewriter=query_rewriter,
    distributed_searcher=base_distributed_searcher,
    K=12,
    k=4,
    use_query_rewrite=True
)
baseline_multi_baseline   = MultiBaselineRetriever(
    rag_system=baseline_rag,
    retriever=baseline_multi_helper,
    query_rewriter=multi_query_rewriter,
    distributed_searcher=base_distributed_searcher,
    K=12,
    k=2,
    use_query_rewrite=True,
    max_queries=3
)

baseline_baseline_metadata = MetadataRetriever(
    rag_system=baseline_rag,
    query_rewriter=baseline_query,
    distributed_searcher=base_distributed_searcher,
    metadata_score=metadata_score,
    K=12,
    k=4,
    use_query_rewrite=True
)

baseline_baseline_crossen = CrossEncoderRetriever(
    rag_system=baseline_rag,
    query_rewriter=baseline_query,
    distributed_searcher=base_distributed_searcher,
    K=12,
    k=4,
    use_query_rewrite=True
)

baseline_baseline_metadatacrossen = MetadataCrossEncoderRetriever(
    rag_system=baseline_rag,
    query_rewriter=baseline_query,
    distributed_searcher=base_distributed_searcher,
    metadata_score=metadata_score,
    cross_encoder_score=cross_encoder_score,
    score_fusion=score_fusion,
    K=12,
    k=4,
    use_query_rewrite=True
)



### combine hypothesis test

In [ ]:
#base,toc
#rewrite,multi,base
#base,metadata,metadatacrossen,crossen


# s1st_baseline_rewriter_baseline = BaselineRetriever(
#     rag_system=baseline_rag,
#     query_rewriter=query_rewriter,
#     distributed_searcher=base_distributed_searcher,
#     K=12,
#     k=4,
#     use_query_rewrite=True
# )

s2nd_toc_rewriter_baseline = BaselineRetriever(
    rag_system=rag,
    query_rewriter=query_rewriter,
    distributed_searcher=base_distributed_searcher,
    K=12,
    k=4,
    use_query_rewrite=True
)

# s2nd_baseline_multi_baseline_helper = BaselineRetriever(
#     rag_system=baseline_rag,
#     query_rewriter=query_rewriter,
#     distributed_searcher=base_distributed_searcher,
#     K=12,
#     k=4,
# )
# s2nd_baseline_multi_baseline = MultiBaselineRetriever(
#     rag_system=baseline_rag,
#     retriever=s2nd_baseline_multi_baseline_helper,
#     query_rewriter=multi_query_rewriter,
#     distributed_searcher=base_distributed_searcher,
#     K=12,
#     k=2,
#     use_query_rewrite=True,
#     max_queries=3
# )

s2nd_baseline_rewriter_metadata= MetadataRetriever(
    rag_system=baseline_rag,
    query_rewriter=query_rewriter,
    distributed_searcher=base_distributed_searcher,
    metadata_score=metadata_score,
    K=12,
    k=4,
    use_query_rewrite=True
)

# hy_helper = MetadataCrossEncoderRetriever(
#     rag_system=rag,
#     query_rewriter=query_rewriter,
#     distributed_searcher=distributed_searcher,
#     metadata_score=metadata_score,
#     cross_encoder_score=cross_encoder_score,
#     score_fusion=score_fusion,
#     K=12,
#     k=4,
#     use_query_rewrite=True
# )
# hy_toc_multi_metadatacrossen = MultiMetadataCrossEncoderRetriever(
#     rag_system=rag,
#     query_rewriter=multi_query_rewriter,
#     retriever=hy_helper,
#     distributed_searcher=distributed_searcher,
#     metadata_score=metadata_score,
#     cross_encoder_score=cross_encoder_score,
#     score_fusion=score_fusion,
#     K=12,
#     k=2,
#     use_query_rewrite=True,
#     max_queries=3
# )

In [ ]:
import pandas as pd
data_q_ref = pd.read_csv("/content/concat_s2nd_toc_rewriter_baseline - GIS_fixed.csv")
data_q_ref = data_q_ref[['question','reference_context']]
data_q_ref

# playground

## retriever

In [ ]:
llm = ChatOpenAI(
    model="qwen/qwen3-vl-30b-a3b-instruct",
    temperature=0.1,
    openai_api_key=OPEN_ROUTER_KEY,  # picked up automatically if set
    base_url="https://openrouter.ai/api/v1",        # only if using OpenRouter-compatible
)


answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an assistant that answers questions for Thai bachelor degree students based on the provided [Context].


**Response Language**: Always respond in Thai language, clear and concise, like a student handbook.

**Rules answer**:
1. Fix Thai spelling: Replace any instance of consonant + space + vowel "อา" with "อำ" instead
   Example: "น า" should be written as "นำ"
2. Start your response with a sentence derived from the question
   Example:
   - Question: "Can you tell me the details about Calculus for Physical Sciences II?"
   - Answer: "The details for Calculus for Physical Sciences II are ____"
3. Keep responses concise, clear, and easy to understand - avoid overly complex language
4. Maintain a polite and respectful tone at all times
5. Answer in Only Thai language
6. DO NOT use asterisks or bullet points in your response
ึ7. DO NOT use (*) character
8. include "ครับ" when speaking in the response.


***Subject Information***: If ask about Subject Information:
- If the context contains complete subject information (subject description and conditions), use this format:
  - <Subject Code> <Subject Name in Thai> <Subject Name in English>
  - <Credits>
  - <Subject conditions>
  - <Subject description> (Briefly, using only keywords)
- If the context does NOT contain subject description or subject conditions, respond like:
  "ขออภัยค่ะ ข้อมูล Subject Code___ ข้อมูลวิชาดังกล่าวไม่เพียงพอครับ"
  Example:
  - Question: "ใน cs วิชา SC 352 002 เรียนเกี่ยวกับอะไร"
  - Context: [does not have about Subject conditions and Subject description]
  - Answer: "ขออภัยค่ะ ข้อมูลวิชา SC 352 002 การออกแบบประสบการณ์ผู้ใช้ ข้อมูลวิชาดังกล่าวไม่เพียงพอครับ"


**Enrollment Plan**: If ask about Enrollment Plan Information use Enrollment Plan format
  - ปีที่ <year> ภาคการศึกษาที่ <semester>
  - <Course Code> <Course Name in Thai only> <credits in format: X(X-X-X)> หน่วยกิต
  - วิชาเลือก/วิชาเลือกเฉพาะทาง (<credit> หน่วยกิต) for elective courses
  - รวม <total credits> หน่วยกิต
"""),
    ("user",
     """Question:
{question}

[Context]
{context}

Respond strictly following the rules above.""")
])

qa_chain = answer_prompt | llm | StrOutputParser()






In [ ]:

query = "ใน AI ต้องเก็บหน่วยกิตของหมวดวิชาศึกษาทั่วไปกี่หน่วยกิต"

print(f"\n{'='*60}")
print(f"Query: {query}")
print('='*60)

retrievers = {
    "baseline_baseline_baseline": baseline_baseline_baseline,
    "toc_baseline_baseline": toc_baseline_baseline,
    "baseline_rewriter_baseline": baseline_rewriter_baseline,
    "baseline_multi_baseline": baseline_multi_baseline,
    "baseline_baseline_metadata": baseline_baseline_metadata,
    "baseline_baseline_crossen": baseline_baseline_crossen,
    "baseline_baseline_metadatacrossen": baseline_baseline_metadatacrossen,
    "s1st_baseline_rewriter_baseline": s1st_baseline_rewriter_baseline,
    "s2nd_toc_rewriter_baseline": s2nd_toc_rewriter_baseline,
    "s2nd_baseline_multi_baseline": s2nd_baseline_multi_baseline,
    "s2nd_baseline_rewriter_crossen": s2nd_baseline_rewriter_crossen,
    "hy_toc_multi_metadatacrossen": hy_toc_multi_metadatacrossen,
}


query = "ใน AI ต้องเก็บหน่วยกิตของหมวดวิชาศึกษาทั่วไปกี่หน่วยกิต"

results = {}
for name, retriever in retrievers.items():
    print(f"\n{'='*60}")
    print(f"Invoking retriever: {name}")
    print(f"{'='*60}")
    try:
        result = retriever.invoke(query)
        results[name] = result
        print(f"Done: {name}")
        print(f"Top Result Sample:\n{result[:2] if isinstance(result, list) else result}\n")
    except Exception as e:
        print(f"Error invoking {name}: {e}")
    print("-"*60)


## ans

In [ ]:
for name, res in results.items():
    print("\n" + "=" * 60)
    print(f"{name}")
    print("=" * 60)

    context_text = "\n\n".join([
        f"{d.metadata.get('title', '')}:{d.metadata.get('course_name', '')}\n{d.page_content}"
        for d in res
    ])

    final_answer = qa_chain.invoke({"question": query, "context": context_text})
    print(final_answer)

In [ ]:
for name, res in results.items():
    print("\n" + "=" * 60)
    context_text = "\n\n".join([
        f"\n\n---Context---\n__{d.metadata.get('course_name', '')}__\n{d.page_content}"
        for d in res
    ])

    print(context_text)

# batch answer

## load data reference

In [ ]:
import pandas as pd

## do all question

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an assistant that answers questions for Thai bachelor degree students based on the provided [Context].


**Response Language**: Always respond in Thai language, clear and concise, like a student handbook.

**Rules answer**:
1. Fix Thai spelling: Replace any instance of consonant + space + vowel "อา" with "อำ" instead
   Example: "น า" should be written as "นำ"
2. Start your response with a sentence derived from the question
   Example:
   - Question: "Can you tell me the details about Calculus for Physical Sciences II?"
   - Answer: "The details for Calculus for Physical Sciences II are ____"
3. Keep responses concise, clear, and easy to understand - avoid overly complex language
4. Maintain a polite and respectful tone at all times
5. Answer in Only Thai language
6. DO NOT use asterisks or bullet points in your response
ึ7. DO NOT use (*) character
8. include "ครับ" when speaking in the response.


***Subject Information***: If ask about Subject Information:
- If the context contains complete subject information (subject description and conditions), use this format:
  - <Subject Code> <Subject Name in Thai> <Subject Name in English>
  - <Credits>
  - <Subject conditions>
  - <Subject description> (Briefly, using only keywords)
- If the context does NOT contain subject description or subject conditions, respond like:
  "ขออภัยค่ะ ข้อมูล Subject Code___ ข้อมูลวิชาดังกล่าวไม่เพียงพอครับ"
  Example:
  - Question: "ใน cs วิชา SC 352 002 เรียนเกี่ยวกับอะไร"
  - Context: [does not have about Subject conditions and Subject description]
  - Answer: "ขออภัยค่ะ ข้อมูลวิชา SC 352 002 การออกแบบประสบการณ์ผู้ใช้ ข้อมูลวิชาดังกล่าวไม่เพียงพอครับ"


**Enrollment Plan**: If ask about Enrollment Plan Information use Enrollment Plan format
  - ปีที่ <year> ภาคการศึกษาที่ <semester>
  - <Course Code> <Course Name in Thai only> <credits in format: X(X-X-X)> หน่วยกิต
  - วิชาเลือก/วิชาเลือกเฉพาะทาง (<credit> หน่วยกิต) for elective courses
  - รวม <total credits> หน่วยกิต
"""),
    ("user",
     """Question:
{question}

[Context]
{context}

Respond strictly following the rules above.""")
])

In [ ]:
import os
import pandas as pd
import numpy as np
from typing import Any, List, Dict, Optional, Protocol
from dataclasses import dataclass

from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI  # or your provider

# --- Protocols ---
class RetrieverProtocol(Protocol):
    def invoke(self, query: str) -> List[Any]: ...

class ChainProtocol(Protocol):
    def invoke(self, inputs: Dict[str, Any]) -> str: ...

# --- Helpers ---
def _safe_get_docs(r: RetrieverProtocol, q: str) -> List[Any]:
    """Call whichever retriever interface exists (invoke / get_relevant_documents / __call__)."""
    for fn in (getattr(r, a, None) for a in ("invoke", "get_relevant_documents", "__call__")):
        if callable(fn):  # type: ignore
            return fn(q)  # type: ignore
    return []

def _ctx(docs: List[Any]) -> str:
    """Build a single context string from docs."""
    print("callling---------------------")
    return "\n\n".join([
        f"\n\n---Context---\n__{getattr(x, 'metadata', {}).get('course_name', '')}__\n{getattr(x, 'page_content', '')}"
        for x in docs
    ])

# --- Config ---
@dataclass
class QAConfig:
    batch_size: int = 15
    max_workers: int = 10
    keep_retrieved: bool = True
    # Optional: set to True to print simple debug statements
    debug: bool = False

# --- LangChain-concurrent Batch Runner (single retrieval shared) ---
class LCConcurrentBatchQA:
    """
    Ensures a single retrieval per input and shares the same docs/context with both:
      1) the LLM (qa_chain)
      2) the saved DataFrame column 'retrieved_docs'
    This eliminates divergence where answers look correct but saved context looks 'missing'.

    MODIFIED: This class now saves a checkpoint file (CSV) for each completed batch
    to the specified `output_dir` instead of returning a single DataFrame at the end.
    """
    def __init__(
        self,
        prefix: str,
        qa_chain: ChainProtocol,        # expects {"question","context"} -> str
        retriever: RetrieverProtocol,
        output_dir: str,                # Directory to save batch checkpoints
        cfg: Optional[QAConfig] = None,
    ):
        self.prefix = prefix
        self.retriever = retriever
        self.cfg = cfg or QAConfig()
        self.output_dir = output_dir

        # Ensure the output directory exists
        os.makedirs(self.output_dir, exist_ok=True)
        if self.cfg.debug:
            print(f"[LCConcurrentBatchQA] Checkpoints will be saved to: {self.output_dir}")

        # 1) Effective question = prefix + raw question (no rewrite)
        compute_q_eff = (
            RunnableLambda(lambda d: d["question"])
            | RunnableLambda(lambda q: f"{self.prefix}{q}")
        )

        # 2) Retrieve DOCS ONCE
        retrieve_docs = (
            compute_q_eff
            | RunnableLambda(lambda q_eff: _safe_get_docs(self.retriever, q_eff))
        )

        # (optional) debug: count docs
        def _dbg_docs(docs: List[Any]) -> List[Any]:
            if self.cfg.debug:
                print(f"[LCConcurrentBatchQA] Retrieved {len(docs)} docs")
            return docs

        retrieve_docs = retrieve_docs | RunnableLambda(_dbg_docs)

        # 3) Build context string from the SAME docs
        docs_to_ctx = RunnableLambda(_ctx)

        # 4) Prepare inputs for LLM using the SAME docs/context
        # We compute both question and docs first, then map to {"question","context"}
        qa_inputs = RunnableParallel(
            question=compute_q_eff,
            docs=retrieve_docs
        ) | RunnableLambda(lambda d: {"question": d["question"], "context": _ctx(d["docs"])})

        # 5) Model chain -> parsed string
        run_llm = qa_inputs | qa_chain | StrOutputParser()

        # 6) Pack outputs (use the SAME docs for retrieved_docs)
        def pack_output(d: Dict[str, Any]) -> Dict[str, Any]:
            original = d["_input"]  # original {"question":..., "reference":...}
            docs = d["docs"]        # SAME docs object used to build LLM context
            out = {
                "question": original.get("question", ""),
                "prediction": d["_pred"],
                "reference": original.get("reference", ""),
                "retrieved_docs": _ctx(docs) if self.cfg.keep_retrieved else "",
            }
            return out

        # 7) Final graph: run both branches in parallel, but retrieval happens only once
        self._graph = (
            RunnableParallel(
                _input=RunnablePassthrough(),  # carries original input row
                docs=retrieve_docs,            # single shared retrieval
                _pred=run_llm,                 # LLM uses the same docs via qa_inputs
            )
            | RunnableLambda(pack_output)
        )

    def _get_processed_info(self) -> (int, int):
        """Checks the output_dir for existing batches and counts processed rows and batches."""
        total_rows = 0
        batch_files = []
        try:
            files = os.listdir(self.output_dir)
            batch_files = sorted([
                f for f in files
                if f.startswith('batch_') and f.endswith('.csv')
            ])
        except FileNotFoundError:
            # Directory doesn't exist yet, so 0 processed
            if self.cfg.debug:
                print(f"[LCConcurrentBatchQA] Output directory not found. Starting from scratch.")
            return 0, 0

        if not batch_files:
            if self.cfg.debug:
                print(f"[LCConcurrentBatchQA] No existing batches found. Starting from scratch.")
            return 0, 0

        for f in batch_files:
            filepath = os.path.join(self.output_dir, f)
            try:
                # We read the file to get its precise row count
                df_batch = pd.read_csv(filepath)
                total_rows += len(df_batch)
            except Exception as e:
                print(f"[LCConcurrentBatchQA] WARNING: Could not read existing batch file {filepath}: {e}. Skipping.")

        num_batches = len(batch_files)
        if self.cfg.debug:
            print(f"[LCConcurrentBatchQA] Found {num_batches} existing batches with {total_rows} total processed rows.")

        return total_rows, num_batches

    def run(self, df: pd.DataFrame) -> None:
        """
        Processes the DataFrame in batches and saves each batch's result
        to a CSV file in the `self.output_dir`. Does not return a DataFrame.
        SKIPS rows that are already processed in existing checkpoint files.
        """
        # --- NEW: Check for existing work ---
        processed_row_count, processed_batch_count = self._get_processed_info()

        original_total_rows = len(df)
        if processed_row_count >= original_total_rows:
            print(f"[LCConcurrentBatchQA] All {original_total_rows} rows are already processed. Nothing to do.")
            return

        if processed_row_count > 0:
            print(f"[LCConcurrentBatchQA] Resuming run. Skipping first {processed_row_count} processed rows.")
            df_to_process = df.iloc[processed_row_count:].reset_index(drop=True)
        else:
            df_to_process = df.reset_index(drop=True)


        # Prepare input rows for the graph
        rows = [
            {
                "question": str(df_to_process.loc[i, "question"]).strip(),
                "reference": str(df_to_process.loc[i, "reference"]) if "reference" in df_to_process.columns else "",
            }
            for i in range(len(df_to_process)) # <-- Use df_to_process
        ]

        if not rows:
            print("[LCConcurrentBatchQA] Input DataFrame is empty or all rows were skipped. No work to do.")
            return

        # Split *remaining* rows into batches
        batches = np.array_split(rows, np.ceil(len(rows) / self.cfg.batch_size))

        if self.cfg.debug:
            print(f"[LCConcurrentBatchQA] Processing {len(rows)} new rows in {len(batches)} batches (file index offset by {processed_batch_count}).")

        for i, batch_rows_np in enumerate(batches):
            batch_rows = list(batch_rows_np) # Convert numpy array elements to list

            if self.cfg.debug:
                print(f"[LCConcurrentBatchQA] --- Starting Batch {i+1}/{len(batches)} ---")

            # Track blanks to preserve alignment with outputs *within this batch*
            blanks_mask = [not r["question"] for r in batch_rows]
            non_blank_inputs = [r for r in batch_rows if r["question"]]

            outputs: List[Dict[str, Any]] = []
            if non_blank_inputs:
                try:
                    outputs = self._graph.batch(
                        non_blank_inputs,
                        config={"max_concurrency": self.cfg.max_workers}
                    )
                except Exception as e:
                    print(f"[LCConcurrentBatchQA] ERROR processing batch {i}. Stopping run.")
                    print(f"[LCConcurrentBatchQA] Error details: {e}")
                    print(f"[LCConcurrentBatchQA] Results for batches 0 to {i-1} are saved.")
                    # Stop processing further batches
                    return

            # Stitch outputs back in original batch order
            final_batch_rows: List[Dict[str, Any]] = []
            it = iter(outputs)
            for is_blank, original in zip(blanks_mask, batch_rows):
                if is_blank:
                    final_batch_rows.append({
                        "question": "",
                        "prediction": "No question",
                        "reference": original.get("reference", ""),
                        "retrieved_docs": "" if not self.cfg.keep_retrieved else "",
                    })
                else:
                    try:
                        final_batch_rows.append(next(it))
                    except StopIteration:
                        # This should not happen if logic is correct
                        print(f"[LCConcurrentBatchQA] WARNING: Mismatch in batch {i} non-blank inputs and outputs.")

            # Save this completed batch to a checkpoint file
            df_batch = pd.DataFrame(final_batch_rows, columns=["question", "prediction", "reference", "retrieved_docs"])

            # --- MODIFIED: Save checkpoint with offset ---
            batch_index = i + processed_batch_count
            batch_filepath = os.path.join(self.output_dir, f"batch_{batch_index:05d}.csv")

            try:
                df_batch.to_csv(batch_filepath, index=False)
                if self.cfg.debug:
                    print(f"[LCConcurrentBatchQA] Successfully saved checkpoint: {batch_filepath}")
            except Exception as e:
                print(f"[LCConcurrentBatchQA] ERROR saving checkpoint {batch_filepath}. Stopping run.")
                print(f"[LCConcurrentBatchQA] Error details: {e}")
                return

        print(f"[LCConcurrentBatchQA] Run finished. All checkpoints saved to {self.output_dir}")

# --- Utility Function to Merge Checkpoints ---

def merge_checkpoints(output_dir: str, output_filepath: str = "merged_results.csv") -> Optional[pd.DataFrame]:
    """
    Finds all 'batch_*.csv' files in `output_dir`, merges them in order,
    saves them to `output_filepath`, and returns the merged DataFrame.
    """
    try:
        files = sorted([
            os.path.join(output_dir, f)
            for f in os.listdir(output_dir)
            if f.startswith('batch_') and f.endswith('.csv')
        ])
    except FileNotFoundError:
        print(f"Error: Output directory not found: {output_dir}")
        return None

    if not files:
        print(f"No batch files (batch_*.csv) found in {output_dir}")
        return pd.DataFrame(columns=["question", "prediction", "reference", "retrieved_docs"])

    print(f"Merging {len(files)} batch files from {output_dir}...")

    df_list = []
    for f in files:
        try:
            df_list.append(pd.read_csv(f))
        except Exception as e:
            print(f"Error reading file {f}: {e}. Skipping this file.")

    if not df_list:
        print("No batch files could be read.")
        return pd.DataFrame(columns=["question", "prediction", "reference", "retrieved_docs"])

    merged_df = pd.concat(df_list, ignore_index=True)

    try:
        merged_df.to_csv(output_filepath, index=False)
        print(f"Successfully merged {len(df_list)} files and saved to {output_filepath}")
    except Exception as e:
        print(f"Error saving merged file to {output_filepath}: {e}")

    return merged_df



llm = ChatOpenAI(
    model="qwen/qwen3-next-80b-a3b-instruct",
    temperature=0.1,
    openai_api_key=OPEN_ROUTER_KEY,
    base_url="https://openrouter.ai/api/v1",
)

qa_chain = answer_prompt | llm | StrOutputParser()

# --- Instantiate the concurrent runner ---


In [ ]:
df_test_ref = pd.read_csv("/content/reference_answer.csv")
df_test_ref.rename(columns={'user_input':'question'}, inplace=True)
df_test_ref=df_test_ref
df_test_ref

In [ ]:
retrievers = {
    # "baseline_baseline_baseline": baseline_baseline_baseline,
    # "toc_baseline_baseline": toc_baseline_baseline,
    # "baseline_rewriter_baseline": baseline_rewriter_baseline,
    # "baseline_multi_baseline": baseline_multi_baseline,
    # "baseline_baseline_metadata": baseline_baseline_metadata,
    # "baseline_baseline_crossen": baseline_baseline_crossen,
    # "baseline_baseline_metadatacrossen": baseline_baseline_metadatacrossen,
    # "s1st_baseline_rewriter_baseline": s1st_baseline_rewriter_baseline,
    "s2nd_toc_rewriter_baseline": s2nd_toc_rewriter_baseline,
    # "s2nd_baseline_multi_baseline": s2nd_baseline_multi_baseline,
    "s2nd_baseline_rewriter_metadata": s2nd_baseline_rewriter_metadata,
    # "hy_toc_multi_metadatacrossen": hy_toc_multi_metadatacrossen,
}
retriever_to_use = "s2nd_toc_rewriter_baseline"

In [ ]:
config = QAConfig(
    batch_size=10,
    max_workers=8,
    debug=True
)

runner = LCConcurrentBatchQA(
    prefix="",
    qa_chain=qa_chain,
    retriever=retrievers[retriever_to_use],
    output_dir=f"./qa_checkpoints/{retriever_to_use}",
    cfg=config
)

In [ ]:
df_chatbot_answer = runner.run(df_test_ref)

In [ ]:
print("\n--- MERGING RESULTS ---")
final_df = merge_checkpoints(
    output_dir=f"./qa_checkpoints/{retriever_to_use}",
    output_filepath=f"qa_results_{retriever_to_use}.csv"
)